# 05 — Similarity

Turn each DayHike in one region into a row of similarity facts, one Facet at a
time, and write that table next to the Catalog. The second half of the
notebook compares DayHikes on those facts. This is the notebook design phase
of ADR-0011 (proposed): production Ranking, the CLI and the Catalog do not
change.

**Region:** DayHikes whose Trailhead lies in lat 42.3–46.9, lon 13.3–19.5
(Slovenia + Croatia: Julian Alps, karst, Velebit, the coast, the Pannonian
lowland). `load_region` reads the GeometryStore the way `04a_geometry.ipynb`
does (vertices, amenities, dayhikes, ways as GeoDataFrames) with the Catalog's
`feature_vectors`, Trailhead and Difficulty merged onto `dayhikes`. Rows are
filtered by region DayHike id in SQL; the full GeometryStore is ~21 GB.

It needs the local Catalog, `data/catalog.sqlite` plus
`data/catalog.geometry.sqlite`, with `sample-dem`, `sample-landcover` and
`sample-climate` run. A missing file, ClimateNormal or FABDEM stops the
notebook with an error; it does not run on the committed OSM fixture.

**Facets here:** Effort, Altitude, Profile, Landscape, Weather, Water, Facilities, Trail,
RoundTrip, Steepness.

A missing fact stays missing (NaN), never 0: a DayHike without elevation is
not a flat DayHike.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import Point
from shapely.ops import substring

from dayhike import geometry_store_path_for
from dayhike.catalog import GAP_JOIN_M
from dayhike.similarity import (
    CLIMATE_MONTH_COLUMNS,
    CORRIDOR_M,
    FACILITIES_FLAG_COLUMNS,
    LANDCOVER_SHARE_COLUMNS,
    SURFACE_SHARE_COLUMNS,
    STEEPEST_KM_WINDOW_M,
    STEEPNESS_STEP_M,
    TRAILHEAD_M,
    WATER_SOURCE_KINDS,
    altitude_facts,
    combined_score,
    corridor_amenities,
    effort_facts,
    facet_score,
    facilities_facts,
    landscape_facts,
    load_region,
    mix,
    multi_seed_score,
    presence,
    profile_dtw,
    profile_facts,
    profile_lockstep,
    ratio,
    round_trip_facts,
    similarity_facts_path_for,
    steepness_facts,
    surface_class,
    surface_class_from_tag,
    tolerance,
    trail_facts,
    water_facts,
    weather_facts,
)


def show_fig(fig):
    try:
        from IPython.display import display

        display(fig)
    except ImportError:
        pass
    plt.close(fig)


repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    if (candidate / "pyproject.toml").is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("pyproject.toml: run the notebook inside the repository")

catalog_path = repo / "data" / "catalog.sqlite"
for path in (catalog_path, geometry_store_path_for(catalog_path)):
    if not path.is_file():
        raise FileNotFoundError(f"{path}: ingest the Catalog first (README)")

REGION_LAT = (42.3, 46.9)
REGION_LON = (13.3, 19.5)
vertices, amenities, dayhikes, ways = load_region(
    catalog_path, lat_range=REGION_LAT, lon_range=REGION_LON
)
if dayhikes[list(CLIMATE_MONTH_COLUMNS)].isna().all().all():
    raise RuntimeError("the Catalog has no ClimateNormal: run sample-climate")
print("catalog", catalog_path)
for name, frame in (
    ("vertices", vertices),
    ("amenities", amenities),
    ("dayhikes", dayhikes),
    ("ways", ways),
):
    print(f"region {name}\t{len(frame)}")
print("name_origin", dayhikes["name_origin"].value_counts().to_dict())

## Effort

How much work the day is. Three facts per DayHike:

- `length_m`: the Catalog length of the main walk.
- `elevation_gain_m`: metres climbed along the walk. OSM way order is
  arbitrary, so a point-to-point DayHike stored "downhill" would show almost no
  climb. The fact is the larger of the climb and the descent along the walk,
  which is the same in both directions. On a loop the two are equal. Vertices
  without `ele` are skipped (the Catalog does the same); no `ele` at all leaves
  the gain missing.
- `difficulty_rank`: Difficulty as the SAC rank 1–6 (`hiking` 1 …
  `difficult_alpine_hiking` 6), from the strictest `sac_scale` on member ways.
  Missing when no member way is tagged, which is common.

In [ ]:
effort = effort_facts(dayhikes, vertices)
print(effort.describe().round(1).to_string())
print("effort missing share")
print(effort.isna().mean().round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(effort["length_m"].dropna() / 1000, bins=40, color="#0f766e")
axes[0].set_xlabel("length (km)")
axes[1].hist(effort["elevation_gain_m"].dropna(), bins=40, color="#c2410c")
axes[1].set_xlabel("elevation gain (m)")
ranks = effort["difficulty_rank"].value_counts().sort_index()
axes[2].bar(ranks.index.astype(int).astype(str), ranks.to_numpy(), color="#7c3aed")
axes[2].set_xlabel("Difficulty rank (SAC)")
for ax in axes:
    ax.set_ylabel("DayHikes")
fig.suptitle("Effort facts in the region")
fig.tight_layout()
show_fig(fig)

## Altitude

How high the DayHike goes, apart from how much it climbs. Two facts, read from
the Catalog's `feature_vectors` (FABDEM when `sample-dem` has run, else OSM
`ele`):

- `highest_m`: the highest vertex.
- `lowest_m`: the lowest vertex. A coastal climb from sea level and a walk from
  a 1500 m pass can reach the same top.

Both are the same whichever way the DayHike is walked.

In [ ]:
altitude = altitude_facts(dayhikes)
print(altitude.describe().round(1).to_string())
print("altitude missing share")
print(altitude.isna().mean().round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 3.5))
bins = 40
ax.hist(altitude["lowest_m"].dropna(), bins=bins, alpha=0.6, color="#1d4ed8", label="lowest")
ax.hist(altitude["highest_m"].dropna(), bins=bins, alpha=0.6, color="#c2410c", label="highest")
ax.set_xlabel("elevation (m)")
ax.set_ylabel("DayHikes")
ax.set_title("Altitude facts in the region")
ax.legend()
fig.tight_layout()
show_fig(fig)

## Profile

The form of the elevation plot, apart from how high the DayHike goes and how
much it climbs. One fact per DayHike, `profile_shape`, its ProfileShape:

1. Cumulative distance along the vertices in stored order: haversine
   metres between consecutive vertices, over every vertex.
2. Elevation at 64 equal steps of the distance fraction 0 → 1, interpolated
   between the vertices that have `ele` (the ElevationProfile, resampled).
3. Subtract the lowest of the 64 values and divide by max(range, 50 m), so
   height is in [0, 1]. A DayHike that varies by less than 50 m is not
   stretched: 5 m of noise stays at most 0.1, and a flat walk looks flat.

Fewer than two vertices with `ele` leaves the ProfileShape missing. The shape
keeps the stored direction; the Profile comparison below takes it forward and
reversed, so OSM's way order does not matter.

The figure picks, for four contrasting forms, the region DayHike with an OSM
name whose ProfileShape is closest (`profile_lockstep`) to a drawn template: a single
summit (up and down), a traverse (one climb end to end), rolling (several
climbs) and flat. Each DayHike is drawn in its stored direction, so a traverse
may run downhill.

In [ ]:
profile = profile_facts(dayhikes, vertices)
shapes = profile["profile_shape"].dropna()
print(f"profile missing share\t{1 - len(shapes) / len(profile):.3f}")
print(f"profile range under 50 m share\t{shapes.map(lambda s: s.max() < 1).mean():.3f}")

fraction = np.linspace(0.0, 1.0, 64)
templates = {
    "single summit": np.interp(fraction, [0.0, 0.5, 1.0], [0.0, 1.0, 0.0]),
    "traverse": fraction,
    "rolling": (1 - np.cos(fraction * 8 * np.pi)) / 2,
    "flat": np.zeros(64),
}
hikes = dayhikes.set_index("id")
names = hikes["name"]
named = shapes[hikes.loc[shapes.index, "name_origin"] == "osm_name"]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True)
for ax, (form, template) in zip(axes, templates.items()):
    closest = named.map(lambda s: profile_lockstep(template, s)).idxmax()
    print(f"profile_example\t{form}\t{names[closest]}")
    ax.plot(fraction, template, color="#a8a29e", linestyle="--", label="template")
    ax.plot(fraction, shapes[closest], color="#0f766e", label="DayHike")
    ax.set_title(f"{form}\n{names[closest]}", fontsize=8)
    ax.set_xlabel("fraction of length")
axes[0].set_ylabel("ProfileShape height")
axes[0].legend(fontsize=8)
fig.tight_layout()
show_fig(fig)

## Landscape

What the DayHike looks like, apart from how hard or how high it is. Nine facts
per DayHike:

- Six LandCover shares (`tree_share`, `shrub_share`, `grass_share`,
  `bare_share`, `snow_share`, `other_share`). The first five are the Catalog's
  `feature_vectors` fractions (each GeometryStore vertex counts once). The
  sixth is the leftover, 1 − their sum, clipped at 0: WorldCover classes the
  Catalog does not keep (cropland, built-up, wetland, water) land there, so
  the six shares always sum to 1 and can be compared as a mix. When the
  Catalog fractions are missing (a hike outside the sampled rasters) all six
  are missing, not 0.
- `peak_summited`: a Peak (`natural=peak` amenity) within 50 m of the line.
- `viewpoint_along`: a Viewpoint (`tourism=viewpoint`) inside the 100 m
  Corridor.
- `peaks_around`: the number of Peaks within 2 km of the line.

**Corridor.** An amenity point is along a DayHike when it is within 100 m of
DayHikeGeometry, measured to the line and not only to the vertices (vertex
spacing reaches 120 m, and a way can have one segment of several hundred
metres). `corridor_amenities(vertices, amenities, kind=..., radius_m=...)`
inserts points along each segment, at most radius / 10 apart, and asks a
haversine BallTree for the amenity points near them. It returns one row per
(DayHike, point) with the distance to the line, so any amenity kind and any
radius work. The Water and Facilities Facets reuse it. All three radii here
(50 m, 100 m, 2 km) use the same helper, so the line is read the same way
whichever way the DayHike is walked.

Peaks and Viewpoints are raw GeometryStore amenities, not Catalog
AmenityFacts: they are looked up around the line, not the Trailhead.

In [ ]:
landscape = landscape_facts(dayhikes, vertices, amenities)
print(landscape.describe().round(3).to_string())
print("landscape missing share")
print(landscape.isna().mean().round(3).to_string())
print(f"peak_summited_hikes\t{int(landscape['peak_summited'].sum())}\tof\t{len(landscape)}")
print(f"viewpoint_along_hikes\t{int(landscape['viewpoint_along'].sum())}\tof\t{len(landscape)}")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
shares = landscape[list(LANDCOVER_SHARE_COLUMNS)].dropna()
colors = ["#15803d", "#84cc16", "#facc15", "#a8a29e", "#38bdf8", "#e7e5e4"]
left = 0.0
for column, color in zip(LANDCOVER_SHARE_COLUMNS, colors):
    width = shares[column].mean()
    axes[0].barh([0], [width], left=left, color=color, edgecolor="white", label=column[:-6])
    left += width
axes[0].set_xlim(0, 1)
axes[0].set_yticks([])
axes[0].set_xlabel("mean share of the DayHike")
axes[0].set_title("LandCover mix (six shares)", fontsize=10)
axes[0].legend(fontsize=7, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.3))
axes[1].hist(shares["tree_share"], bins=20, color="#15803d")
axes[1].set_xlabel("tree share")
axes[1].set_ylabel("DayHikes")
axes[2].hist(landscape["peaks_around"], bins=30, color="#7c3aed")
axes[2].set_xlabel("Peaks within 2 km")
axes[2].set_ylabel("DayHikes")
fig.suptitle("Landscape facts in the region")
fig.tight_layout()
show_fig(fig)

## Weather

What weather the DayHike is likely to have, and how much sun its slopes get.
Forty facts per DayHike:

- The ClimateNormal regime, read from the Catalog's `feature_vectors`
  (WorldClim 2.1, 1970–2000, averaged along DayHikeGeometry):
  `temperature_c` (June–August mean, °C), `precipitation_mm` (mean monthly
  precipitation over the year, mm) and `solar_radiation` (June–August mean,
  kJ m⁻² day⁻¹). This is the slice used when the Hiker names no QueryMonth.
- The twelve months, `temperature_c_01` … `solar_radiation_12`, read from the
  Catalog's `climate_months`. The Weather comparison below has one `QUERY_MONTH` setting that swaps
  the regime for one month's three values (ADR-0010).
- `southness`: SolarAspect, how south-facing the slopes along the DayHike are.
  It is terrain read from the DEM, not weather (shade from trees is
  LandCover). It sits in this Facet because it says how much sun the hike
  gets.

**SolarAspect rule** (`solar_aspect_southness`, FABDEM 1° tiles in `data/dem/`,
ADR-0008):

1. About 32 vertices per DayHike: every k-th vertex counted from the nearer
   end of the walk (k = ⌈vertices / 32⌉), so the same vertices are read whichever
   way the DayHike is walked.
2. FABDEM is read at the vertex and 2 arc-seconds east and north of it. That
   is two FABDEM cells (about 62 m north, 44 m east at 45° N), so the offset
   always reads a different cell, at a known distance.
3. The rise to the east and to the north give the slope and the downslope
   direction. Southness = (1 − the north part of the downslope direction) / 2:
   1 due south, 0 due north, 0.5 east or west, 0.85 south-east or south-west.
4. Points with a slope under 5° are left out. A valley floor or a broad
   ridge top faces no way, and a metre or two of FABDEM noise over 44–62 m
   would give it a random direction. The DayHike's southness is the mean over
   its sloped points. With none (a flat walk) southness is missing, not 0.5.

Notebook `04`'s prototype scored (1 + cos(aspect − π)) / 2 with the aspect
measured from east, which gives 1 for a west-facing slope and 0.5 for a
south-facing one. It is not used here. The rule above is tested on synthetic
north-, south-, east- and west-facing ramp rasters in `tests/test_similarity.py`.

On a mostly flat walk the few sloped points (an embankment, a river bank)
decide southness, so its value says little; it is still a value, because
some slope was found. The maps therefore show the three most north-facing and
the three most south-facing named DayHikes of at least 3 km with at least
500 m between their lowest and highest point, on a topographic basemap (the
line alone when offline). Southness missing on every DayHike means no FABDEM
tile covers the region, and the cell stops with an error.

In [ ]:
dem_dir = catalog_path.parent / "dem"
weather = weather_facts(dayhikes, vertices, dem_dir)
WEATHER_REGIME = ["temperature_c", "precipitation_mm", "solar_radiation", "southness"]
print("dem", dem_dir)
print(weather[WEATHER_REGIME].describe().round(2).to_string())
print("weather missing share")
print(weather[WEATHER_REGIME].isna().mean().round(3).to_string())
print(f"climate_months_missing_share\t{weather[list(CLIMATE_MONTH_COLUMNS)].isna().any(axis=1).mean():.3f}")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
labels = [
    "June–August temperature (°C)",
    "mean monthly precipitation (mm)",
    "June–August solar radiation (kJ m⁻² day⁻¹)",
    "southness (0 north, 1 south)",
]
for ax, column, label, color in zip(axes, WEATHER_REGIME, labels, ["#c2410c", "#1d4ed8", "#ca8a04", "#7c3aed"]):
    ax.hist(weather[column].dropna(), bins=30, color=color)
    ax.set_xlabel(label, fontsize=8)
axes[0].set_ylabel("DayHikes")
fig.suptitle("Weather facts in the region (ClimateNormal regime and SolarAspect)")
fig.tight_layout()
show_fig(fig)

hikes = dayhikes.set_index("id")
facing = weather[["southness"]].join(altitude).join(hikes[["name", "name_origin", "length_m", "geometry"]])
facing = facing[
    (facing["name_origin"] == "osm_name")
    & (facing["length_m"] >= 3_000)
    & (facing["highest_m"] - facing["lowest_m"] >= 500)
].dropna(subset=["southness"])
if weather["southness"].isna().all():
    raise RuntimeError(f"southness missing everywhere: no FABDEM tile in {dem_dir}, run sample-dem")
examples = pd.concat([facing.nsmallest(3, "southness"), facing.nlargest(3, "southness")])
fig, axes = plt.subplots(1, len(examples), figsize=(3 * len(examples), 3.2))
for ax, (dayhike_id, row) in zip(np.atleast_1d(axes), examples.iterrows()):
    print(f"southness_example\t{row['name']}\t{row['southness']:.2f}")
    one = gpd.GeoDataFrame([row], geometry="geometry", crs=dayhikes.crs).to_crs(epsg=3857)
    one.plot(ax=ax, color="#1d4ed8" if row["southness"] < 0.5 else "#c2410c", linewidth=2)
    try:
        import contextily as cx

        cx.add_basemap(ax, source=cx.providers.OpenTopoMap)
    except Exception as error:  # offline: the line alone is still the map
        print("basemap skipped", type(error).__name__)
    ax.set_title(f"{row['name']}\nsouthness {row['southness']:.2f}", fontsize=8)
    ax.set_axis_off()
fig.suptitle("Most north-facing (blue) and most south-facing (orange) named DayHikes")
fig.tight_layout()
show_fig(fig)

## Water

How much water the DayHike passes. Two facts per DayHike:

- `water_count`: the number of WaterSources in the 100 m Corridor, the same band
  as Landscape's Viewpoint: GeometryStore amenities of kind `drinking_water`,
  `spring`, `water_tap` or `water_point` (`WATER_SOURCE_KINDS`).
  The fact is the real count; the Water measure below caps it at three, so "none", "one",
  "a couple" and "plenty" are the levels that matter.
- `dry_stretch_m`: DryStretch, the longest distance walked without passing one.

**DryStretch rule** (`water_facts`):

1. The Corridor points come from `corridor_amenities`, so a point beside the
   middle of a long segment counts, and the distance is to the line.
2. Each point is placed at its position along the DayHike: the place on the
   walk closest to it, in metres from the first vertex (haversine along every
   vertex). Where a path passes the same fountain twice, the closer pass
   counts.
3. The walk is cut at those positions. Stretches run from the first vertex to the
   first point, between points, and from the last point to the last vertex. The
   longest is the DryStretch. Two points at one position cut the walk once.
4. With no point in the Corridor the DryStretch is the length of the walk.
   A water point at the very start or end cuts nothing: a hike whose
   only fountain is at the car park still has a DryStretch of almost its whole
   length.

The positions are measured along the line, so reversing the walk turns every
position into length minus position and leaves the DryStretch alone.

**What counts as water.** Ingest keeps OSM `amenity=drinking_water`,
`natural=spring`, `man_made=water_tap` and `amenity=water_point`, and drops a
spring, tap or water point tagged `drinking_water=no` (CONTEXT.md WaterSource).
A spring or tap that is also tagged `amenity=drinking_water` stays a
drinking-water point, so the Catalog's drinking-water AmenityFact is the same
as before springs were kept. Streams, troughs and unmapped springs are still
not counted, so a backcountry hike can look drier than it is.

The code below prints the region's WaterSources by kind and the split by how
high the DayHike goes. The region box holds 10,325 drinking-water points,
5,759 springs, 617 taps and 190 water points. Springs moved the mountains most:
55% of the DayHikes that reach above 1,500 m pass no WaterSource, 62% of those
reaching 500 to 1,500 m and 55% of those under 500 m. When only drinking water
counted, on the Catalog before the main walk, those shares were 73%, 74% and
65%. Of the region, 58% of the DayHikes pass no WaterSource (71% with drinking
water alone before the main walk).

In [ ]:
water = water_facts(dayhikes, vertices, amenities)
print(water.describe().round(1).to_string())
print("water missing share")
print(water.isna().mean().round(3).to_string())
for kind in WATER_SOURCE_KINDS:
    print(f"water_source_in_region_box\t{kind}\t{int((amenities['kind'] == kind).sum())}")
print(f"water_zero_share\t{(water['water_count'] == 0).mean():.3f}")
print(f"water_three_or_more_share\t{(water['water_count'] >= 3).mean():.3f}")

bands = pd.cut(
    altitude["highest_m"], [-1, 500, 1500, 10_000], labels=["under 500 m", "500-1500 m", "over 1500 m"]
)
by_band = water.groupby(bands, observed=True).agg(
    hikes=("water_count", "size"),
    no_water_share=("water_count", lambda c: (c == 0).mean()),
    median_dry_stretch_km=("dry_stretch_m", lambda d: d.median() / 1000),
)
print("water by highest point")
print(by_band.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
levels = water["water_count"].clip(upper=6).value_counts().sort_index()
axes[0].bar(levels.index.astype(int).astype(str), levels.to_numpy(), color="#1d4ed8")
axes[0].set_xticks(range(len(levels)), [str(int(i)) if i < 6 else "6+" for i in levels.index])
axes[0].set_xlabel("WaterSources in the Corridor")
axes[0].set_ylabel("DayHikes")
axes[1].hist(water["dry_stretch_m"].dropna() / 1000, bins=40, color="#0f766e")
axes[1].set_xlabel("DryStretch (km)")
axes[1].set_ylabel("DayHikes")
fig.suptitle("Water facts in the region")
fig.tight_layout()
show_fig(fig)

# One named DayHike with its WaterSources: all of them nearby in grey, the
# Corridor ones in blue, the line in orange.
hikes = dayhikes.set_index("id")
example_id = hikes.index[hikes["name"] == "Kanjon Velika Paklenica"][0]
line = hikes.loc[[example_id]]
chain = vertices[vertices["dayhike_id"] == example_id]
spots = pd.concat(
    corridor_amenities(chain, amenities, kind=kind, radius_m=CORRIDOR_M) for kind in WATER_SOURCE_KINDS
)
print(f"example_dayhike\t{example_id}\t{hikes.at[example_id, 'name']}")
print(water.loc[[example_id]].round(1).to_string())
box = line.total_bounds
pad = 0.01
near = amenities[
    amenities["kind"].isin(WATER_SOURCE_KINDS)
    & amenities["longitude"].between(box[0] - pad, box[2] + pad)
    & amenities["latitude"].between(box[1] - pad, box[3] + pad)
]
fig, ax = plt.subplots(figsize=(6, 6))
line.to_crs(epsg=3857).plot(ax=ax, color="#c2410c", linewidth=2)
near[~near["osm_key"].isin(spots["osm_key"])].to_crs(epsg=3857).plot(ax=ax, color="#a8a29e", markersize=25)
near[near["osm_key"].isin(spots["osm_key"])].to_crs(epsg=3857).plot(ax=ax, color="#1d4ed8", markersize=45)
try:
    import contextily as cx

    cx.add_basemap(ax, source=cx.providers.OpenTopoMap)
except Exception as error:  # offline: the line alone is still the map
    print("basemap skipped", type(error).__name__)
ax.set_title(
    f"{hikes.at[example_id, 'name']}: {int(water.at[example_id, 'water_count'])} WaterSources in the Corridor "
    f"(blue), DryStretch {water.at[example_id, 'dry_stretch_m'] / 1000:.1f} km",
    fontsize=9,
)
ax.set_axis_off()
fig.tight_layout()
show_fig(fig)

## Facilities

What the DayHike offers along the way and where it starts. Five yes/no flags
per DayHike:

- Along the DayHike, in the 100 m Corridor: `hut_along` (a Hut),
  `restaurant_cafe_along` (a restaurant or cafe), `shelter_along` (a Shelter).
- At the Trailhead, within 500 m of the Catalog Trailhead:
  `restaurant_cafe_trailhead`, `parking_trailhead`.

**Rules** (`facilities_facts`):

1. Along is the Corridor of the Landscape section: `corridor_amenities` finds
   the amenity points within 100 m of the line (distance to the line, not only
   to the vertices), here for the kinds `hut`, `restaurant_cafe` and `shelter`.
   Reversing the walk changes nothing.
2. At the Trailhead is a plain distance: the point is within 500 m (haversine)
   of the Catalog Trailhead (`trailhead_latitude` and `trailhead_longitude`),
   which is where the main walk starts, near parking when any lies within
   5 km. 450 m counts, 550 m does not. The production
   AmenityFacts count around the Trailhead in a 20 km disk; that disk is not
   used here, so a cafe in the next village does not make the Trailhead a
   meal stop.
3. A Shelter (`amenity=shelter`, a lean-to or weather shelter) is a separate
   kind and never sets the Hut flag, and a Hut never sets the Shelter flag.
4. A flag is plain false when the amenity is not there. Two hikes that both lack
   a Hut agree on that point, and the Facilities measure below counts that as agreement. The
   flags have no missing state: a DayHike that is not in the data has no row at
   all.

**Data notes.** The restaurant or cafe flags only know what OSM maps, which is
thin in the mountains and thicker in towns and vineyards. The Trailhead is
chosen near parking when any lies within 5 km, so `parking_trailhead` is true more often
than a random point on the map would give. The code below prints how often each
flag is set, how the number of set flags is spread, and how the flags change
with the highest point.

**Region result** (4,584 DayHikes, no flag is ever missing): a Hut along 22.8%,
a restaurant or cafe along 33.9%, a Shelter along 17.0%, a restaurant or cafe at
the Trailhead 37.3%, parking at the Trailhead 59.3%. 24.5% of the DayHikes set
no flag, 24.7% one, 18.7% two, 21.2% three, 9.9% four and 0.9% all five. The
flags follow the terrain the way one would expect: a Hut is along 4% of the
DayHikes that stay under 500 m, 25% of those reaching 500 to 1,500 m and 41% of
those above 1,500 m, while a restaurant or cafe along the way falls from 44% to
33% to 23% and parking at the Trailhead from 65% to 63% to 45%.

In [ ]:
facilities = facilities_facts(dayhikes, vertices, amenities)
print("facilities missing share")
print(facilities.isna().mean().round(3).to_string())
for flag in FACILITIES_FLAG_COLUMNS:
    print(f"{flag}\t{int(facilities[flag].sum())}\tof\t{len(facilities)}\t{facilities[flag].mean():.3f}")
flag_count = facilities.sum(axis=1)
print("flags_set_share\t" + str(flag_count.value_counts(normalize=True).sort_index().round(3).to_dict()))

bands = pd.cut(
    altitude["highest_m"], [-1, 500, 1500, 10_000], labels=["under 500 m", "500-1500 m", "over 1500 m"]
)
print("share of DayHikes with each flag, by highest point")
print(facilities.groupby(bands, observed=True).mean().round(3).T.to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
shares = facilities.mean()
axes[0].barh([c.replace("_", " ") for c in shares.index], shares.to_numpy(), color="#7c3aed")
axes[0].invert_yaxis()
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("share of region DayHikes")
counts = flag_count.value_counts().reindex(range(6), fill_value=0)
axes[1].bar(counts.index.astype(str), counts.to_numpy(), color="#0f766e")
axes[1].set_xlabel("flags set (of 5)")
axes[1].set_ylabel("DayHikes")
fig.suptitle("Facilities facts in the region")
fig.tight_layout()
show_fig(fig)

# One named DayHike: the line in orange, the Trailhead and its 500 m disc in
# black, the counted amenities in colour (grey: nearby but not counted).
hikes = dayhikes.set_index("id")
example_name = "Šmartno - Koča pod Kremžarjevim vrhom"
example_id = hikes.index[hikes["name"] == example_name][0]
line = hikes.loc[[example_id]]
print(f"example_dayhike\t{example_id}\t{hikes.at[example_id, 'name']}")
print(facilities.loc[[example_id]].to_string())
trailhead = gpd.GeoSeries.from_xy(
    line["trailhead_longitude"], line["trailhead_latitude"], crs="EPSG:4326"
)
metric = line.estimate_utm_crs()
disc = trailhead.to_crs(metric).buffer(TRAILHEAD_M).to_crs(epsg=3857)
box = line.total_bounds
pad = 0.01
kinds = ("hut", "restaurant_cafe", "shelter", "parking")
near = amenities[
    amenities["kind"].isin(kinds)
    & amenities["longitude"].between(box[0] - pad, box[2] + pad)
    & amenities["latitude"].between(box[1] - pad, box[3] + pad)
]
chain = vertices[vertices["dayhike_id"] == example_id]
along_keys = {
    kind: set(corridor_amenities(chain, amenities, kind=kind, radius_m=CORRIDOR_M)["osm_key"])
    for kind in ("hut", "restaurant_cafe", "shelter")
}
from_trailhead = near.to_crs(metric).distance(trailhead.to_crs(metric).iloc[0])
counted = near.apply(
    lambda row: row["osm_key"] in along_keys.get(row["kind"], ())
    or (row["kind"] in ("restaurant_cafe", "parking") and from_trailhead[row.name] <= TRAILHEAD_M),
    axis=1,
)
colors = {"hut": "#b91c1c", "restaurant_cafe": "#1d4ed8", "shelter": "#15803d", "parking": "#7c3aed"}
fig, ax = plt.subplots(figsize=(6, 6))
line.to_crs(epsg=3857).plot(ax=ax, color="#c2410c", linewidth=2)
disc.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1.2)
if (~counted).any():
    near[~counted].to_crs(epsg=3857).plot(ax=ax, color="#a8a29e", markersize=20)
for kind, color in colors.items():
    chosen = near[counted & (near["kind"] == kind)]
    if len(chosen):
        chosen.to_crs(epsg=3857).plot(ax=ax, color=color, markersize=55, label=kind.replace("_", "/"))
try:
    import contextily as cx

    cx.add_basemap(ax, source=cx.providers.OpenTopoMap)
except Exception as error:  # offline: the line alone is still the map
    print("basemap skipped", type(error).__name__)
ax.legend(fontsize=8, loc="best")
ax.set_title(
    f"{hikes.at[example_id, 'name']}: {int(flag_count[example_id])} of 5 flags set (circle: 500 m round the Trailhead)",
    fontsize=9,
)
ax.set_axis_off()
fig.tight_layout()
show_fig(fig)

## Trail

What the DayHike walks on, apart from where it is. Five facts per DayHike,
from its walked member ways (`in_chain = 1`; Leftovers are not walked):

- Four SurfaceMix shares (`paved_share`, `gravel_share`, `natural_share`,
  `rock_share`): the length fraction of each class, length being the metres of
  the way on the walk (`walked_m`). They sum to 1 over the ways that could be classified.
- `tarmac_approach_m`: TarmacApproach. Start at the Catalog Trailhead, where
  the walk starts, and add up the walked length of consecutive paved ways until
  the first way that is not paved. It is 0 when the first way is unpaved, and the
  whole classified length when every way is paved. The ways are taken in the
  stored walk order (`walk_order`, not member order), and a loop counts the
  shorter of its two directions, so the walking direction does not matter.

**Classifying a way.** The `surface` tag decides when it is one of these
(first `;` token, lowercased):

| Class | `surface` values |
| --- | --- |
| paved | asphalt, concrete (and `concrete:*`), paving_stones, sett, cobblestone, paved |
| gravel | compacted, gravel, fine_gravel, unpaved, pebblestone |
| natural | ground, dirt, earth, grass, mud, sand, woodchips |
| rock | rock, bare_rock, stone |

A way with no `surface`, or an unlisted one (`wood`, `metal`, ...), is
classified from its `highway`: roads (motorway down to residential, service,
living_street, pedestrian, cycleway, unclassified, and their `_link` forms)
are paved; a `track` follows its `tracktype` (grade1 paved, grade2-3 gravel,
grade4-5 natural, none gravel); `path`, `footway`, `steps` and `bridleway` are
natural; `via_ferrata` is rock. Any other way is left out of the denominator
(and ends the TarmacApproach). A DayHike with no classified way has all five
facts missing, not 0.

In [ ]:
trail = trail_facts(dayhikes, vertices, ways)
print(trail.describe().round(3).to_string())
print("trail missing share")
print(trail.isna().mean().round(3).to_string())

walked = ways[ways["in_chain"] == 1].drop_duplicates("way_id")
bags = [json.loads(tags) for tags in walked["tags"]]
by_tag = sum(surface_class_from_tag(bag) is not None for bag in bags)
by_fallback = sum(
    surface_class_from_tag(bag) is None and surface_class(bag) is not None for bag in bags
)
print(
    f"ways_classified\tby_tag={by_tag}\tby_highway_fallback={by_fallback}\t"
    f"left_out={len(bags) - by_tag - by_fallback}\tof\t{len(bags)}"
)
print(
    "fallback_highways",
    Counter(
        bag.get("highway")
        for bag in bags
        if surface_class_from_tag(bag) is None and surface_class(bag) is not None
    ).most_common(8),
)
print(f"tarmac_zero_share\t{(trail['tarmac_approach_m'] == 0).mean():.3f}")

SURFACE_COLORS = {"paved": "#475569", "gravel": "#d97706", "natural": "#15803d", "rock": "#a8a29e"}
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
mean_mix = trail[list(SURFACE_SHARE_COLUMNS)].mean()
left = 0.0
for kind, column in zip(SURFACE_COLORS, SURFACE_SHARE_COLUMNS):
    axes[0].barh([0], [mean_mix[column]], left=left, color=SURFACE_COLORS[kind], edgecolor="white", label=kind)
    left += mean_mix[column]
axes[0].set_xlim(0, 1)
axes[0].set_yticks([])
axes[0].set_xlabel("mean share of the DayHike")
axes[0].set_title("SurfaceMix (four shares)", fontsize=10)
axes[0].legend(fontsize=7, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.3))
axes[1].hist(trail["paved_share"].dropna(), bins=20, color=SURFACE_COLORS["paved"])
axes[1].set_xlabel("paved share")
axes[1].set_ylabel("DayHikes")
axes[2].hist(trail["tarmac_approach_m"].dropna().clip(upper=6_000) / 1_000, bins=30, color="#7c3aed")
axes[2].set_xlabel("TarmacApproach (km, clipped at 6)")
axes[2].set_ylabel("DayHikes")
fig.suptitle("Trail facts in the region")
fig.tight_layout()
show_fig(fig)

# One DayHike with a road approach and three or more surfaces, ways coloured by class.
classes_present = (trail[list(SURFACE_SHARE_COLUMNS)] > 0.05).sum(axis=1)
candidates = trail[(trail["tarmac_approach_m"] > 0) & (classes_present >= 3)]
example_id = int(candidates.index[0])
example = ways[(ways["dayhike_id"] == example_id) & (ways["in_chain"] == 1)].copy()
example["surface_class"] = [surface_class(json.loads(tags)) or "unknown" for tags in example["tags"]]
example = example[example.geometry.notna()]
print(f"example_dayhike\t{example_id}\t{dayhikes.set_index('id').at[example_id, 'name']}")
print(trail.loc[[example_id]].round(3).to_string())
fig, ax = plt.subplots(figsize=(6, 5))
for kind, color in {**SURFACE_COLORS, "unknown": "#dc2626"}.items():
    part = example[example["surface_class"] == kind]
    if len(part):
        part.plot(ax=ax, color=color, linewidth=3, label=kind)
ax.legend(fontsize=8)
ax.set_title(f"Ways of DayHike {example_id} by surface class", fontsize=10)
fig.tight_layout()
show_fig(fig)

## RoundTrip

Whether the DayHike's length is the whole walk back to the car. One fact per
DayHike:

- `round_trip`: True when the last vertex of the main walk lies within 50 m of
  the first, the Trailhead. 50 m is `GAP_JOIN_M`, the gap at which two member
  ways count as connected.

**Rule** (`round_trip_facts`):

1. The haversine distance between the first and the last vertex of
   DayHikeGeometry. Swapping the ends leaves it alone, so the walking direction
   does not matter.
2. 50 m or less is a RoundTrip: walking `length_m` brings the Hiker back to the
   Trailhead.
3. A lollipop is not a RoundTrip. The main walk of a loop with a stem
   (ADR-0012) starts at the stem's far end, walks the stem and the loop, and
   stops where the stem meets the loop. To get back the Hiker walks the stem
   again, which `length_m` does not hold.
4. A line is not a RoundTrip. OSM does not say whether it is walked
   point-to-point (a shuttle or a bus back) or out-and-back (about twice
   `length_m`); its length is one way either way.
5. The fact is never missing: every DayHike has a main walk.

The Trail Facet's TarmacApproach uses the same 50 m test to decide when the
walk may start from either end.

The code below prints how many region DayHikes are RoundTrips, lollipops and
lines (a lollipop here: the walk ends within 50 m of its own line, leaving out
the last 200 m), draws the distance between the ends on a log scale, and maps
one of each kind with its first vertex (green) and last vertex (red).

**Region result** (4,584 DayHikes, never missing): 386 RoundTrips (8.4%), 186
lollipops (4.1%) and 4,012 lines (87.5%). The 50 m cut sits in a gap: 366
DayHikes end within 10 m of where they start, only 20 more within 50 m, 30 more
within 100 m. A line's ends are a median 67% of its length apart, so most lines
are point-to-point or out-and-back, not near misses. For nine in ten region
DayHikes `length_m` is therefore not the whole walk back to the car
(`docs/future-work.md`, "Walking back on a line").

In [ ]:
round_trip = round_trip_facts(dayhikes, vertices)
print("round_trip missing share")
print(round_trip.isna().mean().round(3).to_string())

hikes = dayhikes.set_index("id")
lines = hikes.geometry.to_crs(dayhikes.estimate_utm_crs())


def end_gap_m(line):
    return Point(line.coords[0]).distance(Point(line.coords[-1]))


def ends_on_own_walk(line):
    return line.length > 400 and Point(line.coords[-1]).distance(substring(line, 0, line.length - 200)) <= GAP_JOIN_M


end_gap = lines.map(end_gap_m)
kind = pd.Series("line", index=hikes.index)
kind[lines.map(ends_on_own_walk)] = "lollipop"
kind[round_trip["round_trip"]] = "round_trip"
for name in ("round_trip", "lollipop", "line"):
    print(f"walk_kind\t{name}\t{int((kind == name).sum())}\tof\t{len(kind)}\t{(kind == name).mean():.3f}")
for metres in (10, 50, 100, 500):
    print(f"ends_within\t{metres} m\t{int((end_gap <= metres).sum())}")
gap_share = (end_gap / hikes["length_m"])[kind == "line"]
print(f"line_end_gap_share_of_length\tmedian={gap_share.median():.2f}")

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(np.log10(end_gap.clip(lower=1)), bins=40, color="#0f766e")
ax.axvline(np.log10(GAP_JOIN_M), color="#c2410c", linestyle="--", label="50 m")
ax.set_xticks([0, 1, 2, 3, 4], ["1 m", "10 m", "100 m", "1 km", "10 km"])
ax.set_xlabel("distance between first and last vertex (log)")
ax.set_ylabel("DayHikes")
ax.set_title("RoundTrip facts in the region")
ax.legend()
fig.tight_layout()
show_fig(fig)

# One named DayHike of each kind: the line, its first vertex (green, the
# Trailhead) and its last vertex (red).
EXAMPLES = {
    "round_trip": "Po stopinjah pastirjev",
    "lollipop": "Arriacher Hofwanderweg-Variante",
    "line": "Velika planina - Planina Dol",
}
examples = {name: hikes.index[hikes["name"] == example_name][0] for name, example_name in EXAMPLES.items()}
fig, axes = plt.subplots(1, len(examples), figsize=(4 * len(examples), 4), squeeze=False)
for ax, (name, example_id) in zip(axes[0], examples.items()):
    print(f"example_dayhike\t{name}\t{example_id}\t{hikes.at[example_id, 'name']}\tend_gap_m={end_gap[example_id]:.0f}")
    one = hikes.loc[[example_id]].to_crs(epsg=3857)
    one.plot(ax=ax, color="#c2410c", linewidth=2)
    coords = one.geometry.iloc[0].coords
    ax.scatter(*coords[0], s=60, color="#15803d", zorder=5)
    ax.scatter(*coords[-1], s=30, color="#b91c1c", zorder=6)
    try:
        import contextily as cx

        cx.add_basemap(ax, source=cx.providers.OpenTopoMap)
    except Exception as error:  # offline: the line alone is still the map
        print("basemap skipped", type(error).__name__)
    ax.set_title(f"{name}: {hikes.at[example_id, 'name']}\nends {end_gap[example_id]:.0f} m apart", fontsize=8)
    ax.set_axis_off()
fig.tight_layout()
show_fig(fig)

## Steepness

How steep the climbing is, apart from how long the DayHike is and how much it
climbs in all. Effort holds the totals; Steepness lets the Hiker ask for "as
steep as this one" at any length. Two facts per DayHike:

- `climb_grade` (ClimbGrade): the elevation gain over the length of the
  stretches that climb, a fraction (0.12 is 12 %).
- `steepest_km_m` (SteepestKm): the largest height change, up or down, over any
  1 km of the walk, in metres. The hardest sustained stretch of the day.

**Rule** (`steepness_facts`):

1. Elevation is resampled every ~50 m (`STEEPNESS_STEP_M`) of haversine
   distance along the main walk, interpolated between the vertices that have
   `ele`. Mappers place vertices 5 m or 50 m apart; on raw vertices the DEM's
   metre-scale jitter between close vertices adds climb, so resampling keeps
   both facts the same for any vertex spacing.
2. ClimbGrade takes the steps that rise in the direction with the larger climb
   (the same choice as Effort's `elevation_gain_m`) and divides their rise by
   their length. Flat and falling stretches do not dilute it, so a RoundTrip,
   which walks back down, and a line up the same slope score alike. Plain
   gain / length would halve the RoundTrip: on a walk that goes up and
   straight back down it is half the ClimbGrade.
3. SteepestKm is the largest |height change| between two resampled points 1 km
   (`STEEPEST_KM_WINDOW_M`) apart. Taking the absolute value keeps it the same
   in both walking directions. A walk shorter than 1 km uses its two ends.
4. Fewer than two vertices with `ele` leaves both missing; a flat walk is 0,
   not missing.

The share of length steeper than 30 % was also tried: it ranks the region
almost the same as ClimbGrade (Spearman 0.88), so it is not a third fact.

The code below prints the spread, ClimbGrade against plain gain / length
(Effort's `elevation_gain_m` over `length_m`) by RoundTrip, and the named
DayHikes at the 5th, 50th and 95th ClimbGrade percentile.

**Region result** (4,584 DayHikes, never missing): ClimbGrade has a median of
12 % (middle half 8–17 %, 95th percentile 27 %) and SteepestKm a median of
149 m (middle half 91–224 m, 95th percentile 349 m). Plain gain / length is a
median 0.66 of ClimbGrade on RoundTrips against 0.89 on the other DayHikes, so
it would rank a RoundTrip as gentler than a line up the same slope. (Both
ratios sit above the ideal 0.5 and 1 because Effort's raw-vertex gain carries
DEM jitter, `docs/future-work.md`, "Smoothed elevation gain".) The steepest
DayHikes are via ferratas and summit paths, with ClimbGrade up to 79 %.

In [ ]:
steepness = steepness_facts(dayhikes, vertices)
print(steepness.describe().round(3).to_string())
print("steepness missing share")
print(steepness.isna().mean().round(3).to_string())
print(f"steepness_step_m\t{STEEPNESS_STEP_M}\twindow_m\t{STEEPEST_KM_WINDOW_M}")

plain_grade = effort["elevation_gain_m"] / effort["length_m"]
by_kind = pd.DataFrame(
    {
        "climb_grade": steepness["climb_grade"],
        "plain_grade": plain_grade,
        "round_trip": round_trip["round_trip"],
    }
).dropna()
for is_round, group in by_kind.groupby("round_trip"):
    print(
        f"plain_over_climb_grade\tround_trip={is_round}\t{len(group)}"
        f"\tmedian={(group['plain_grade'] / group['climb_grade'].clip(lower=1e-9)).median():.2f}"
    )
named_grades = steepness["climb_grade"][hikes["name_origin"] == "osm_name"].dropna()
for quantile in (0.05, 0.5, 0.95):
    example_id = (named_grades - named_grades.quantile(quantile)).abs().idxmin()
    print(
        f"steepness_example\tp{round(quantile * 100)}\t{hikes.at[example_id, 'name']}"
        f"\tclimb_grade={steepness.at[example_id, 'climb_grade']:.3f}"
        f"\tsteepest_km_m={steepness.at[example_id, 'steepest_km_m']:.0f}"
    )

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(steepness["climb_grade"].dropna() * 100, bins=40, color="#0f766e")
axes[0].set_xlabel("ClimbGrade (%)")
axes[0].set_ylabel("DayHikes")
axes[1].hist(steepness["steepest_km_m"].dropna(), bins=40, color="#c2410c")
axes[1].set_xlabel("SteepestKm (m over 1 km)")
axes[1].set_ylabel("DayHikes")
for is_round, color in ((False, "#a8a29e"), (True, "#7c3aed")):
    group = by_kind[by_kind["round_trip"] == is_round]
    axes[2].scatter(
        group["climb_grade"] * 100, group["plain_grade"] * 100, s=4, color=color,
        label="RoundTrip" if is_round else "not a RoundTrip",
    )
axes[2].plot([0, 60], [0, 60], color="#57534e", linewidth=0.8, linestyle="--")
axes[2].set_xlim(0, 60)
axes[2].set_ylim(0, 60)
axes[2].set_xlabel("ClimbGrade (%)")
axes[2].set_ylabel("gain / length (%)")
axes[2].legend(fontsize=8)
fig.suptitle("Steepness facts in the region")
fig.tight_layout()
show_fig(fig)

## Similarity facts table

One row per region DayHike, indexed by DayHike id: `name`, `name_origin`, the
Trailhead and the DayHike line (for the maps below), then every Facet's facts.
It is written as a pandas pickle next to the Catalog,
`{catalog}.similarity_facts.pkl` (`data/catalog.similarity_facts.pkl` for the
local Catalog). It is regenerable and gitignored (`*.similarity_facts.pkl`).

In [ ]:
facts = gpd.GeoDataFrame(
    dayhikes.set_index("id")[
        ["name", "name_origin", "trailhead_latitude", "trailhead_longitude", "geometry"]
    ]
    .join(effort)
    .join(altitude)
    .join(profile)
    .join(landscape)
    .join(weather)
    .join(water)
    .join(facilities)
    .join(trail)
    .join(round_trip)
    .join(steepness),
    geometry="geometry",
    crs=dayhikes.crs,
)
facts_path = similarity_facts_path_for(catalog_path)
facts.to_pickle(facts_path)
print("similarity_facts", facts_path)
print(f"similarity_facts_rows\t{len(facts)}")
print("columns", list(facts.columns))

# Facet similarity

Compare DayHikes on Facets. Each Facet gives a FacetScore in [0, 1] for a pair
of DayHikes: 1 alike, 0 unlike. The Hiker's FacetWeights combine them into one
score with a coverage figure. Production Ranking stays ADR-0003 cosine.

The facts are the `facts` table built above. After the ten Facets: one Seed
ranked under two FacetWeights sets, a RoundTrip Seed with and without weight
on RoundTrip, a steep Seed with and without weight on Steepness, several
Seeds (the maximum rule), how often each Facet is missing, the verification
summary, and the design points for review.

Each Facet section: the measures and their constants with the region's spread,
then verification. A verification states the expected verdict and the reason
before the score is computed, then prints a grep-able line
`verify<TAB><facet><TAB><check><TAB><score><TAB>PASS|FAIL`.

## Measure kinds and combining

All functions are in `dayhike.similarity`. Each returns a value in [0, 1], or
None when either DayHike lacks the fact.

| Kind | Rule | Used for |
| --- | --- | --- |
| `ratio` | min(a′, b′) / max(a′, b′), a′ = max(a, floor) | sizes: 15 vs 20 km is closer than 5 vs 10 km; ClimbGrade and SteepestKm (Steepness) |
| `tolerance` | max(0, 1 − abs(a − b) / D) | levels: 1800 vs 2000 m is close, 300 vs 2000 m is not |
| `mix` | Σ min(pᵢ, qᵢ) over fractions summing to 1 | LandCover (Landscape), SurfaceMix (Trail) |
| `presence` | share of yes/no flags that agree | Peak summited, Viewpoint along (Landscape), the five Facilities flags, RoundTrip |

- FacetScore (`facet_score`): the plain mean of the Facet's measures that are
  not missing. A missing Difficulty drops out; length and gain still count.
- Combined score (`combined_score`): Σ wᶠ·sᶠ / Σ wᶠ over Facets with a
  FacetScore and wᶠ > 0. FacetWeights can be in any scale (50/30/20 or 5/3/2);
  weight 0 switches a Facet off. Coverage is the scored weight over all positive
  weight. No scored Facet gives a missing score.
- Several Seeds (`multi_seed_score`): a candidate's best combined score to any
  one Seed, never a mix of Seeds.

`measure_spread` prints a measure's spread in the region and, for a fixed
sample of random pairs, the median score its constant gives.

In [ ]:
rng = np.random.default_rng(0)
left_ids = rng.choice(facts.index.to_numpy(), 2000)
right_ids = rng.choice(facts.index.to_numpy(), 2000)


def measure_spread(column, measure, constant):
    values = facts[column]
    q = values.quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(1).tolist()
    pair_scores = [
        measure(facts.at[a, column], facts.at[b, column])
        for a, b in zip(left_ids, right_ids)
    ]
    pair_scores = [s for s in pair_scores if s is not None]
    median = float(np.median(pair_scores))
    print(
        f"measure\t{column}\t{constant}\tmissing={values.isna().mean():.3f}\t"
        f"q05,q25,q50,q75,q95={q}\trandom_pair_median={median:.2f}"
    )

### Pair verification helpers

- `pair_rows` looks up the named DayHikes (each must be a unique `osm_name`) in
  the order similar A, similar B, dissimilar A, dissimilar B.
- `show_pairs` prints their facts and draws each DayHike line on a small
  topographic map (the line alone when offline).
- `verify_pairs` scores both pairs. Both `verify` lines PASS when the similar
  pair scores higher than the dissimilar one. The soft targets (similar ≥ 0.7,
  dissimilar ≤ 0.4) are printed on separate `soft` lines. Both kinds are kept
  for the verification summary at the end.

In [ ]:
def pair_rows(similar_names, dissimilar_names):
    names = [*similar_names, *dissimilar_names]
    named = facts[(facts["name_origin"] == "osm_name") & facts["name"].isin(names)]
    assert sorted(named["name"]) == sorted(set(names)), "pair names must be unique osm_name DayHikes"
    return named.set_index("name").loc[names]


def show_pairs(pairs, columns, title):
    print(pairs[columns].round(1).to_string())
    fig, axes = plt.subplots(1, len(pairs), figsize=(3 * len(pairs), 3))
    for ax, (name, row) in zip(axes, pairs.iterrows()):
        one = gpd.GeoDataFrame([row], geometry="geometry", crs=facts.crs).to_crs(epsg=3857)
        one.plot(ax=ax, color="#c2410c", linewidth=2)
        try:
            import contextily as cx

            cx.add_basemap(ax, source=cx.providers.OpenTopoMap)
        except Exception as error:  # offline: the line alone is still the map
            print("basemap skipped", type(error).__name__)
        ax.set_title(name, fontsize=8)
        ax.set_axis_off()
    fig.suptitle(title)
    fig.tight_layout()
    show_fig(fig)


VERIFY = {}  # (facet, check) -> (score, PASS or FAIL), read by the summary at the end
SOFT = {}  # (facet, check) -> (score, target, met)


def verify(facet, check, score, ok):
    result = "PASS" if ok else "FAIL"
    VERIFY[(facet, check)] = (score, result)
    print(f"verify\t{facet}\t{check}\t{score:.3f}\t{result}")


def verify_pairs(facet, score_fn, pairs):
    similar = score_fn(pairs.iloc[0], pairs.iloc[1])
    dissimilar = score_fn(pairs.iloc[2], pairs.iloc[3])
    ok = similar > dissimilar
    verify(facet, "similar_pair", similar, ok)
    verify(facet, "dissimilar_pair", dissimilar, ok)
    for check, score, target, met in (
        ("similar_pair", similar, ">=0.7", similar >= 0.7),
        ("dissimilar_pair", dissimilar, "<=0.4", dissimilar <= 0.4),
    ):
        SOFT[(facet, check)] = (score, target, met)
        print(f"soft\t{facet}\t{check}\t{score:.3f}\t{target}\t{'met' if met else 'missed'}")

### Synthetic check helpers

`synthetic_checks` runs three checks for one Facet and prints one `verify`
line each:

- identity: the lowest score of a DayHike against itself, which must be 1.
- symmetry: the largest gap between A vs B and B vs A over the random pairs,
  which must be 0.
- reversed: the lowest score of a DayHike's facts against the same DayHike's
  facts rebuilt from its vertices in reverse order (`reversed_vertices`),
  which must be 1. It runs on a small Julian Alps box (Trailhead lat
  46.3–46.5, lon 13.6–14.0) to keep the second pass over the vertices short.

In [ ]:
def reversed_vertices(vertices):
    last = vertices.groupby("dayhike_id")["seq"].transform("max")
    return vertices.assign(seq=last - vertices["seq"])


def synthetic_checks(facet, score_fn, forward, backward):
    """Identity, symmetry and reversed direction for one Facet."""
    identity = [score_fn(row, row) for _, row in facts.iterrows()]
    identity = [s for s in identity if s is not None]
    worst = min(identity)
    verify(facet, "identity", worst, abs(worst - 1.0) < 1e-9)

    gaps = []
    for a, b in zip(left_ids, right_ids):
        ab, ba = score_fn(facts.loc[a], facts.loc[b]), score_fn(facts.loc[b], facts.loc[a])
        if ab is not None:
            gaps.append(abs(ab - ba))
    gap = max(gaps)
    verify(facet, "symmetry_max_gap", gap, gap < 1e-9)

    turned = [score_fn(forward.loc[i], backward.loc[i]) for i in forward.index]
    turned = [s for s in turned if s is not None]
    worst = min(turned)
    verify(facet, "reversed", worst, abs(worst - 1.0) < 1e-9)


CHECK_LAT = (46.3, 46.5)
CHECK_LON = (13.6, 14.0)
in_check = dayhikes["trailhead_latitude"].between(*CHECK_LAT) & dayhikes["trailhead_longitude"].between(*CHECK_LON)
check_dayhikes = dayhikes[in_check]
check_vertices = vertices[vertices["dayhike_id"].isin(check_dayhikes["id"])]
check_ways = ways[ways["dayhike_id"].isin(check_dayhikes["id"])]
backward_vertices = reversed_vertices(check_vertices)
print(f"check_dayhikes\t{len(check_dayhikes)}")

## Effort

Three measures: length, elevation gain, Difficulty rank.

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| `length_m` | ratio | floor 1 km | The Catalog length band starts at 1 km, so the floor only matters for a hand-made input. 6 km vs 18 km scores 0.33. |
| `elevation_gain_m` | ratio | floor 100 m | About 7.5% of region DayHikes climb under 100 m. Those are flat walks and should match each other (10 m vs 30 m scores 1), not sit at the far end of a ratio. |
| `difficulty_rank` | tolerance | D = 3 | Tagged on about half of the region, mostly SAC 1–2. One grade apart scores 0.67; T2 vs T5 scores 0. |

The data does not argue for other values, so the PRD starting constants stay.

In [ ]:
LENGTH_FLOOR_M = 1_000
GAIN_FLOOR_M = 100
DIFFICULTY_D = 3


def effort_score(a, b):
    return facet_score(
        [
            ratio(a["length_m"], b["length_m"], floor=LENGTH_FLOOR_M),
            ratio(a["elevation_gain_m"], b["elevation_gain_m"], floor=GAIN_FLOOR_M),
            tolerance(a["difficulty_rank"], b["difficulty_rank"], d=DIFFICULTY_D),
        ]
    )


measure_spread("length_m", lambda a, b: ratio(a, b, floor=LENGTH_FLOOR_M), f"ratio floor={LENGTH_FLOOR_M}")
measure_spread("elevation_gain_m", lambda a, b: ratio(a, b, floor=GAIN_FLOOR_M), f"ratio floor={GAIN_FLOOR_M}")
measure_spread("difficulty_rank", lambda a, b: tolerance(a, b, d=DIFFICULTY_D), f"tolerance D={DIFFICULTY_D}")

### Effort verification (real region)

Expected before computing:

- **Similar:** "Zagorje - Završje - Kum" (Slovenia) and "Mosor-Kosa" (Dalmatia).
  Both are about 7.4 km at SAC 2, with 750–1,250 m of climbing: a solid
  half-day up a hill. They are far apart and look nothing alike, which Effort
  ignores.
- **Dissimilar:** "Pot kralja Triglava" and "Log - Mangart". The first is a
  4 km, 300 m T1 walk; the second an 11 km, 2,800 m climb at T6.

In [ ]:
effort_pairs = pair_rows(
    ["Zagorje - Završje - Kum", "Mosor-Kosa"],
    ["Pot kralja Triglava", "Log - Mangart"],
)
show_pairs(effort_pairs, ["length_m", "elevation_gain_m", "difficulty_rank"], "Effort pairs")
verify_pairs("effort", effort_score, effort_pairs)

### Effort synthetic checks

- identity: every DayHike against itself scores 1 (worst case printed).
- symmetry: A vs B equals B vs A on 2,000 random pairs (largest gap printed).
- reversed: Effort facts rebuilt from the vertices in reverse order score 1
  against the stored direction (worst case printed). This is the check that keeps
  elevation gain direction-free.

In [ ]:
synthetic_checks(
    "effort",
    effort_score,
    effort_facts(check_dayhikes, check_vertices),
    effort_facts(check_dayhikes, backward_vertices),
)

## Altitude

Two measures: highest and lowest point.

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| `highest_m` | tolerance | D = 1000 m | Region tops run from the coast to about 2,800 m; the middle half spans about 450–1,450 m. 1,800 vs 2,000 m scores 0.8; a lowland hill vs a 2,500 m ridge scores 0. |
| `lowest_m` | tolerance | D = 1000 m | Lowest points run from sea level to about 2,400 m; the middle half spans about 240–890 m. A climb from the sea vs a walk from a 1,500 m pass scores 0. |

A random pair of region DayHikes differs by about 600 m on the highest point
and about 400 m on the lowest, so D = 1000 m puts a typical random pair near
0.4 and 0.6 and keeps pairs from the same kind of terrain high. The PRD
starting constants stay.

In [ ]:
HIGHEST_D_M = 1_000
LOWEST_D_M = 1_000


def altitude_score(a, b):
    return facet_score(
        [
            tolerance(a["highest_m"], b["highest_m"], d=HIGHEST_D_M),
            tolerance(a["lowest_m"], b["lowest_m"], d=LOWEST_D_M),
        ]
    )


measure_spread("highest_m", lambda a, b: tolerance(a, b, d=HIGHEST_D_M), f"tolerance D={HIGHEST_D_M}")
measure_spread("lowest_m", lambda a, b: tolerance(a, b, d=LOWEST_D_M), f"tolerance D={LOWEST_D_M}")

### Altitude verification (real region)

Expected before computing:

- **Similar:** "Vršič - Razor" and "Kotovo sedlo - Mangart". Both start from a
  high Julian Alps road (about 1,600–1,700 m) and top out at about 2,500 m.
- **Dissimilar:** "Omiš - Kula" and "Dolič - Triglav". Omiš starts at the
  Dalmatian coast and tops out below 900 m; the Triglav summit route starts
  above 2,100 m and reaches the highest point in the region.

In [ ]:
altitude_pairs = pair_rows(
    ["Vršič - Razor", "Kotovo sedlo - Mangart"],
    ["Omiš - Kula", "Dolič - Triglav"],
)
show_pairs(altitude_pairs, ["highest_m", "lowest_m"], "Altitude pairs")
verify_pairs("altitude", altitude_score, altitude_pairs)

### Altitude synthetic checks

Same three checks. Highest and lowest point do not depend on the walking
direction, so the reversed check confirms the facts are read the same way.

In [ ]:
synthetic_checks(
    "altitude",
    altitude_score,
    altitude_facts(check_dayhikes),
    altitude_facts(check_dayhikes),
)

## Profile

One measure: the ProfileShape (`profile_shape`, built above: 64 points
along the distance fraction, height over max(range, 50 m)). Two ways to compare
two ProfileShapes, both taking the better of the candidate walked forward or
reversed:

| Method | Rule | Constant |
| --- | --- | --- |
| lock-step (`profile_lockstep`) | 1 − mean abs difference, point i against point i | 64 points, range floor 50 m |
| banded DTW (`profile_dtw`) | 1 − mean abs difference along the cheapest warping path | Sakoe–Chiba band 10% of the points (6) |

DTW lets a climb at 30% of the length line up with a climb at 50%, as far as
the band allows. It never scores below lock-step on the same pair, because the
diagonal is one of its paths. The question is whether that slack separates
similar from dissimilar pairs better, or only lifts every score.

Both scores stay in [0, 1] because every height does. Below, the spread of
each method over the same random pairs as the other Facets.

In [ ]:
profile_methods = {"lockstep": profile_lockstep, "dtw": profile_dtw}
for method, fn in profile_methods.items():
    pair_scores = [
        fn(facts.at[a, "profile_shape"], facts.at[b, "profile_shape"])
        for a, b in zip(left_ids, right_ids)
    ]
    pair_scores = [s for s in pair_scores if s is not None]
    q = np.quantile(pair_scores, [0.05, 0.25, 0.5, 0.75, 0.95]).round(2).tolist()
    print(
        f"measure\tprofile_shape\t{method}\tmissing={facts['profile_shape'].isna().mean():.3f}\t"
        f"random_pair_q05,q25,q50,q75,q95={q}"
    )

### Profile verification (real region)

Expected before computing. Three pairs that should be similar:

- "Koča pri Savici - Dom na Komni" and "Zavala - Sv. Ante": each is one
  straight climb from bottom to top, but 1,400 m over 6.4 km in the Julian
  Alps against 530 m over 2.5 km up from the sea on Hvar. Both are stored
  bottom to top; before the main walk Zavala was stored top to bottom and
  matched only walked reversed.
- "Učna pot škrata Bisera" and "Maslinovi puti": both go up to one top and
  back down. The first tops out about three fifths of the way along, the
  second halfway. This is the pair DTW is meant for.
- "Šturmovci - Ptujsko jezero" and "Iter Aquileiense - variante della Roggia":
  both flat (under 10 m of climbing). A few metres of noise must not make them
  look different.

Three that should not:

- "Koča pri Savici - Dom na Komni" and "Nacionalni park Sjeverni Velebit II -
  Rožanski kukovi": one climb against a 14 km Velebit route that climbs and
  drops several times.
- "Maslinovi puti" and "Blejski razgledni trail": one top against the Bled
  viewpoint trail, which climbs to several viewpoints in turn.
- "Šturmovci - Ptujsko jezero" and "Zavala - Sv. Ante": flat against a
  straight climb.

Three pairs were re-picked after the main walk (ADR-0012) changed their
walks: "Gozdna učna pot Planina–Mirna gora" is now a loop that starts at its
top, "P16" an 11.9 km walk with 540 m of climbing, and "Dom Paklenica - Jurline"
a 3.9 km walk with one climb. None plays its old part.

The table scores all six pairs with both methods. A method separates better
when its margin (lowest similar score minus highest dissimilar score) is
larger; the mean gap between the two groups is printed too. The `verify` lines
below use the kept method on the first pair of each group: the same Komna climb
against another single climb and against a rolling route.

In [ ]:
PROFILE_PAIRS = [
    ("similar", "Koča pri Savici - Dom na Komni", "Zavala - Sv. Ante"),
    ("similar", "Učna pot škrata Bisera", "Maslinovi puti"),
    ("similar", "Šturmovci - Ptujsko jezero", "Iter Aquileiense - variante della Roggia"),
    ("dissimilar", "Koča pri Savici - Dom na Komni", "Nacionalni park Sjeverni Velebit II - Rožanski kukovi"),
    ("dissimilar", "Maslinovi puti", "Blejski razgledni trail"),
    ("dissimilar", "Šturmovci - Ptujsko jezero", "Zavala - Sv. Ante"),
]

profile_rows = pair_rows(sorted({name for _, a, b in PROFILE_PAIRS for name in (a, b)}), [])
profile_table = pd.DataFrame(
    {
        "expected": expected,
        "a": a,
        "b": b,
        **{
            method: fn(profile_rows.at[a, "profile_shape"], profile_rows.at[b, "profile_shape"])
            for method, fn in profile_methods.items()
        },
    }
    for expected, a, b in PROFILE_PAIRS
)
print(profile_table.round(3).to_string())
for method in profile_methods:
    by_group = profile_table.groupby("expected")[method]
    margin = by_group.min()["similar"] - by_group.max()["dissimilar"]
    gap = by_group.mean()["similar"] - by_group.mean()["dissimilar"]
    print(f"profile_separation\t{method}\tmargin={margin:.3f}\tmean_gap={gap:.3f}")

fraction = np.linspace(0.0, 1.0, 64)
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharey=True)
for ax, (expected, a, b) in zip(axes.flat, PROFILE_PAIRS):
    for name, color in ((a, "#0f766e"), (b, "#c2410c")):
        ax.plot(fraction, profile_rows.at[name, "profile_shape"], color=color, label=name)
    ax.set_title(expected, fontsize=9)
    ax.set_xlabel("fraction of length")
    ax.legend(fontsize=7)
fig.suptitle("Profile pairs: ProfileShape in stored direction")
fig.tight_layout()
show_fig(fig)

### Profile method kept: lock-step

The design keeps lock-step. The rule was: DTW only if it separates the
verification pairs better. On the real region it does not (printed above):

| Method | Lowest similar | Highest dissimilar | Margin | Mean gap | Random-pair median |
| --- | --- | --- | --- | --- | --- |
| lock-step | 0.875 | 0.702 | 0.173 | 0.307 | 0.77 |
| banded DTW | 0.975 | 0.813 | 0.161 | 0.255 | 0.86 |

DTW does what it is for on the shifted-top pair (Škrata Bisera against
Maslinovi puti: 0.875 → 0.975). It lifts the dissimilar pairs more on average
(+0.09 against +0.04 for the similar ones): the single top against the Bled
viewpoint trail goes 0.693 → 0.813, because
the band lets one climb of the rolling route line up with the single climb.
Every random pair moves up too, so DTW squeezes the useful range of the Facet
toward 1. The Profile FacetScore below (`profile_score`) is lock-step;
`profile_dtw` stays in `dayhike.similarity` for this comparison.

Profile scores sit high in general (a random pair's median is 0.77): two
ProfileShapes that both go up and down share much of the unit square. The
soft target dissimilar ≤ 0.4 is therefore out of reach: even flat against a
straight climb scores 0.52. The verdict rests on similar > dissimilar.

In [ ]:
def profile_score(a, b):
    return facet_score([profile_lockstep(a["profile_shape"], b["profile_shape"])])


profile_pairs = pair_rows(
    ["Koča pri Savici - Dom na Komni", "Zavala - Sv. Ante"],
    ["Koča pri Savici - Dom na Komni", "Nacionalni park Sjeverni Velebit II - Rožanski kukovi"],
)
show_pairs(profile_pairs, ["length_m", "elevation_gain_m", "highest_m", "lowest_m"], "Profile pairs")

# The ProfileShapes the score compares: the second of each pair in the direction that matches better.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, (a, b), label in zip(axes, [(0, 1), (2, 3)], ["similar", "dissimilar"]):
    shape_a, shape_b = profile_pairs.iloc[a]["profile_shape"], profile_pairs.iloc[b]["profile_shape"]
    if np.mean(np.abs(shape_a - shape_b[::-1])) < np.mean(np.abs(shape_a - shape_b)):
        shape_b = shape_b[::-1]
    ax.plot(fraction, shape_a, color="#0f766e", label=profile_pairs.index[a])
    ax.plot(fraction, shape_b, color="#c2410c", label=profile_pairs.index[b])
    ax.set_title(f"{label} pair: {profile_score(profile_pairs.iloc[a], profile_pairs.iloc[b]):.3f}", fontsize=9)
    ax.set_xlabel("fraction of length")
    ax.legend(fontsize=7)
axes[0].set_ylabel("ProfileShape height")
fig.suptitle("Profile pairs: ProfileShapes")
fig.tight_layout()
show_fig(fig)
verify_pairs("profile", profile_score, profile_pairs)

### Profile synthetic checks

Same three checks. For reversed, the ProfileShape is rebuilt from the vertices
in reverse order, which mirrors it; taking the better of forward and reversed must
give 1.

In [ ]:
synthetic_checks(
    "profile",
    profile_score,
    profile_facts(check_dayhikes, check_vertices),
    profile_facts(check_dayhikes, backward_vertices),
)

## Landscape

Four measures: the LandCover mix, Peak summited, Viewpoint along, Peaks around.
The facts are built above (six LandCover shares including the leftover;
Peak within 50 m; Viewpoint in the 100 m Corridor; Peaks within 2 km, all
measured to the line).

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| LandCover mix (six shares) | mix | none | Scores the shared share: half forest and half grass against all forest is 0.5. The leftover share keeps hikes through cropland or towns from looking like forest hikes with a few missing percent (it averages 7.4%, 95th percentile 33%). A random pair of region DayHikes shares a median 0.69, because forest is 66% of the average hike; forest against bare rock is near 0. |
| `peak_summited` | presence | none | About three in ten region DayHikes pass a Peak. Both-absent counts as agreement, so two hikes with no summit agree on this point. |
| `viewpoint_along` | presence | none | About a quarter of region DayHikes have a Viewpoint in the Corridor; same rule. |
| `peaks_around` | ratio | floor 1 | Counts run from 0 to about 90 (middle half 4–18). Ratio, not a difference, so 20 vs 40 Peaks is as far apart as 2 vs 4. The floor of 1 treats "none" as "one" against a ratio, so 0 vs 3 scores 0.33, not 0. A random pair's median is 0.35. |

The Landscape FacetScore (`landscape_score`) is the plain mean of the four. The
PRD starting constant (floor 1) stays; the data does not argue for another.
Presence measures lift every score: two hikes that both have no summit and no
Viewpoint share 2 of the 4 points before the LandCover mix and Peak count are
compared, which is why the dissimilar pairs below do not reach 0.

Result on the real region (below): the main similar pair scores 0.94 and the
dissimilar one 0.26. Forest against open karst ("Zahum Idovac") scores 0.58:
the mixes share 0.04, but neither hike has a summit or Viewpoint, and they
agree on that. The soft target dissimilar ≤ 0.4 is met for forest against
alpine, not for forest against karst, which is the presence effect above, not
a LandCover error. The verdict rests on similar > dissimilar.

In [ ]:
PEAKS_AROUND_FLOOR = 1
LANDCOVER = list(LANDCOVER_SHARE_COLUMNS)


def landscape_score(a, b):
    return facet_score(
        [
            mix([a[c] for c in LANDCOVER], [b[c] for c in LANDCOVER]),
            presence([a["peak_summited"]], [b["peak_summited"]]),
            presence([a["viewpoint_along"]], [b["viewpoint_along"]]),
            ratio(a["peaks_around"], b["peaks_around"], floor=PEAKS_AROUND_FLOOR),
        ]
    )


mix_scores = [
    mix([facts.at[a, c] for c in LANDCOVER], [facts.at[b, c] for c in LANDCOVER])
    for a, b in zip(left_ids, right_ids)
]
mix_scores = [s for s in mix_scores if s is not None]
q = np.quantile(mix_scores, [0.05, 0.25, 0.5, 0.75, 0.95]).round(2).tolist()
print(
    f"measure\tlandcover_mix\tmix\tmissing={facts[LANDCOVER].isna().any(axis=1).mean():.3f}\t"
    f"random_pair_q05,q25,q50,q75,q95={q}"
)
for column in LANDCOVER:
    print(f"share\t{column}\tmean={facts[column].mean():.3f}\tq95={facts[column].quantile(0.95):.3f}")
for flag in ("peak_summited", "viewpoint_along"):
    print(f"measure\t{flag}\tpresence\ttrue_share={facts[flag].mean():.3f}")
measure_spread("peaks_around", lambda a, b: ratio(a, b, floor=PEAKS_AROUND_FLOOR), f"ratio floor={PEAKS_AROUND_FLOOR}")

### Landscape verification (real region)

Expected before computing:

- **Similar:** "Begunje - Cemarovnik - Velika Špička" and "Rakitna - Vrbica".
  Both are forested (86–93% tree) hikes to a Peak with a Viewpoint on the way,
  among a couple of dozen Peaks. They are far apart, which Landscape ignores.
- **Dissimilar:** "Planinarska staza 16" and "Gorjanska pot". The first is a
  77% forest path with no Peak and no Viewpoint nearby; the second is 99%
  bare rock to a summit among more than 40 Peaks.

In [ ]:
landscape_pairs = pair_rows(
    ["Begunje - Cemarovnik - Velika Špička", "Rakitna - Vrbica"],
    ["Planinarska staza 16", "Gorjanska pot"],
)
show_pairs(
    landscape_pairs,
    [*LANDCOVER, "peak_summited", "viewpoint_along", "peaks_around"],
    "Landscape pairs",
)
verify_pairs("landscape", landscape_score, landscape_pairs)

### More Landscape pairs (real region)

Expected before computing. Two that should be similar, two that should not:

- **Similar:** "Predmeja - Iztokova koča" and "Mošćenice-Kremenjak/Brgud": two
  forest walks (95–97% tree), no summit, no Viewpoint, a dozen or two Peaks
  around. "Dolič - Triglav" and "Špremova pot": two bare-rock summit routes in
  the Julian Alps (94–100% bare, 20–50 Peaks around).
- **Dissimilar:** the forest walk "Predmeja - Iztokova koča" against the bare
  rock route "Dolič - Triglav" (forest vs alpine), and against "Zahum Idovac"
  (97% grass, open karst, few Peaks).

The check is the same as the main pair: the lowest similar score must be above
the highest dissimilar one.

In [ ]:
LANDSCAPE_MORE = [
    ("similar", "Predmeja - Iztokova koča", "Mošćenice-Kremenjak/Brgud"),
    ("similar", "Dolič - Triglav", "Špremova pot"),
    ("dissimilar", "Predmeja - Iztokova koča", "Dolič - Triglav"),
    ("dissimilar", "Predmeja - Iztokova koča", "Zahum Idovac"),
]

more_rows = pair_rows(sorted({n for _, a, b in LANDSCAPE_MORE for n in (a, b)}), [])
more_table = pd.DataFrame(
    {
        "expected": expected,
        "a": a,
        "b": b,
        "landcover_mix": mix([more_rows.at[a, c] for c in LANDCOVER], [more_rows.at[b, c] for c in LANDCOVER]),
        "score": landscape_score(more_rows.loc[a], more_rows.loc[b]),
    }
    for expected, a, b in LANDSCAPE_MORE
)
print(more_table.round(3).to_string())
by_group = more_table.groupby("expected")["score"]
margin = by_group.min()["similar"] - by_group.max()["dissimilar"]
verify("landscape", "more_pairs_margin", margin, margin > 0)

### Landscape synthetic checks

Same three checks. For reversed, the Landscape facts are rebuilt from the
vertices in reverse order; the Corridor and every radius are measured to the line,
so the walking direction must not matter.

In [ ]:
synthetic_checks(
    "landscape",
    landscape_score,
    landscape_facts(check_dayhikes, check_vertices, amenities),
    landscape_facts(check_dayhikes, backward_vertices, amenities),
)

## Weather

Four measures: temperature, precipitation and solar radiation for the slice
that `QUERY_MONTH` selects, and SolarAspect southness. The facts are built
above (the ClimateNormal regime from `feature_vectors`, the twelve months
from `climate_months`, southness from FABDEM: the downslope direction at about
32 vertices, 1 due south, 0 due north, 0.5 east or west, slopes under 5° left
out).

`QUERY_MONTH` is the one QueryMonth setting (ADR-0010). `None` compares the
June–August regime: June–August mean temperature, mean monthly precipitation
over the year, June–August solar radiation. A month 1–12 compares that month's
three values instead. Southness does not change with the month.

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| temperature (°C) | tolerance | D = 10 °C | June–August temperature runs from about 10 °C on the high Julian Alps to 22 °C on the Dalmatian coast (5th–95th percentile). D = 10 °C is that spread: an alpine hut against a coastal walk scores near 0, two hikes in the same valley near 1. A random pair's median is about 0.7. |
| precipitation (mm) | ratio | floor 10 mm | A size, so a ratio: 60 vs 120 mm is as far apart as 100 vs 200 mm. Regime values run 52–168 mm (5th–95th percentile); the wettest Julian Alps get three times the rain of the Pannonian plain. No region month is under 19 mm, so the floor only guards a hand-made input. |
| solar radiation (kJ m⁻² day⁻¹) | tolerance | D = 5,000 | From the region's spread: the June–August regime runs 17,800–22,700 (5th–95th percentile), so D = 5,000 is that spread. A random pair's median is 0.70. The same D is kept for single months. Winter radiation differs less in absolute terms (January 3,900–5,900), so January solar scores sit higher; that is the climate, not a different scale. |
| southness | tolerance | D = 1 | Already in [0, 1]. Region middle half 0.37–0.71, median 0.54; a random pair's median is 0.78. North against south (0 vs 1) scores 0, east against south 0.5. Missing on about 1% of region DayHikes, all flat lowland walks with no point steeper than 5°; it drops out of the mean there. |

The Weather FacetScore (`weather_score`) is the plain mean of the four. PRD
starting constants stay; the solar-radiation D is set from the regime spread
as above. Southness is one measure in four, so two hikes with the same climate
on opposite faces of one mountain still score about 0.7: the climate is the
same, only the sun differs.

In [ ]:
QUERY_MONTH = None  # 1-12 compares that month's ClimateNormal; None the June-August regime
TEMPERATURE_D_C = 10
PRECIPITATION_FLOOR_MM = 10
SOLAR_RADIATION_D = 5_000  # kJ m-2 day-1, the region's 5th-95th percentile spread of the regime
SOUTHNESS_D = 1


def climate_slice(month):
    """Temperature, precipitation and solar radiation columns for a QueryMonth."""
    suffix = "" if month is None else f"_{month:02d}"
    return [f"{name}{suffix}" for name in ("temperature_c", "precipitation_mm", "solar_radiation")]


def weather_score(a, b, month=QUERY_MONTH):
    temperature, precipitation, solar = climate_slice(month)
    return facet_score(
        [
            tolerance(a[temperature], b[temperature], d=TEMPERATURE_D_C),
            ratio(a[precipitation], b[precipitation], floor=PRECIPITATION_FLOOR_MM),
            tolerance(a[solar], b[solar], d=SOLAR_RADIATION_D),
            tolerance(a["southness"], b["southness"], d=SOUTHNESS_D),
        ]
    )


temperature, precipitation, solar = climate_slice(QUERY_MONTH)
print(f"query_month\t{QUERY_MONTH}")
measure_spread(temperature, lambda a, b: tolerance(a, b, d=TEMPERATURE_D_C), f"tolerance D={TEMPERATURE_D_C}")
measure_spread(precipitation, lambda a, b: ratio(a, b, floor=PRECIPITATION_FLOOR_MM), f"ratio floor={PRECIPITATION_FLOOR_MM}")
measure_spread(solar, lambda a, b: tolerance(a, b, d=SOLAR_RADIATION_D), f"tolerance D={SOLAR_RADIATION_D}")
measure_spread("southness", lambda a, b: tolerance(a, b, d=SOUTHNESS_D), f"tolerance D={SOUTHNESS_D}")
for column in ("solar_radiation", "solar_radiation_01"):
    q05, q95 = facts[column].quantile([0.05, 0.95])
    print(f"spread\t{column}\tq05={q05:.0f}\tq95={q95:.0f}\tq95-q05={q95 - q05:.0f}")
print(f"monthly_precipitation_min\t{facts[[f'precipitation_mm_{m:02d}' for m in range(1, 13)]].min().min():.1f}")

### Weather verification (real region)

Expected before computing:

- **Similar:** "Kopiščarjeva pot" (the north face of Prisojnik, Julian Alps)
  and "Frischaufova pot" (Kamnik–Savinja Alps). Two shady, north-facing high
  alpine routes, both around 10 °C in summer, about 155 mm of rain a month and
  the region's lowest summer sun. They are 60 km apart in different ranges,
  which Weather ignores.
- **Dissimilar:** "Kopiščarjeva pot" and "Zavala - Sv. Ante" (Hvar). The
  island climb faces south from the sea, is 23 °C in summer with 62 mm of rain
  a month and the strongest sun in the region.

In [ ]:
weather_pairs = pair_rows(
    ["Kopiščarjeva pot", "Frischaufova pot"],
    ["Kopiščarjeva pot", "Zavala - Sv. Ante"],
)
show_pairs(weather_pairs, [*climate_slice(QUERY_MONTH), "southness"], "Weather pairs")
verify_pairs("weather", weather_score, weather_pairs)

### More Weather pairs (real region)

Expected before computing. Two that should be similar, two that should not:

- **Similar:** "Zavala - Sv. Ante" and "Baška Voda-magistrala(Novaci)": two
  sunny, south-facing Dalmatian coast walks (about 23 °C, 62 and 72 mm of rain
  a month). "Kopiščarjeva pot" and "Koča v Krnici - Križ": two north-facing
  Julian Alps routes.
- **Dissimilar:** "Kopiščarjeva pot" against "Slovenska pot": the same
  mountain (Prisojnik), the same climate, opposite faces (north against
  south). Only southness differs, so it should score clearly below the
  similar pairs but not near 0 (about 0.7). And the north-facing "Koča v
  Grohotu - Raduha" (Kamnik–Savinja Alps) against "Planinarska staza
  Medvednica 3", a south-facing hill above Zagreb, 8 °C warmer with about half
  the rain.

Both similar pairs were re-picked after the main walk (ADR-0012) changed their
walks: "Karst in Kornati" is now a flat 2 km loop with no slope steep enough
for southness, and "Koča pri izviru Soče - Bavški Grintavec" a 6 km walk that
faces no one way (southness 0.42).

The check is the same as the main pair: the lowest similar score must be above
the highest dissimilar one.

In [ ]:
WEATHER_MORE = [
    ("similar", "Zavala - Sv. Ante", "Baška Voda-magistrala(Novaci)"),
    ("similar", "Kopiščarjeva pot", "Koča v Krnici - Križ"),
    ("dissimilar", "Kopiščarjeva pot", "Slovenska pot"),
    ("dissimilar", "Koča v Grohotu - Raduha", "Planinarska staza Medvednica 3"),
]

more_rows = pair_rows(sorted({n for _, a, b in WEATHER_MORE for n in (a, b)}), [])
more_table = pd.DataFrame(
    {
        "expected": expected,
        "a": a,
        "b": b,
        "southness": tolerance(more_rows.at[a, "southness"], more_rows.at[b, "southness"], d=SOUTHNESS_D),
        "score": weather_score(more_rows.loc[a], more_rows.loc[b]),
    }
    for expected, a, b in WEATHER_MORE
)
print(more_table.round(3).to_string())
by_group = more_table.groupby("expected")["score"]
margin = by_group.min()["similar"] - by_group.max()["dissimilar"]
verify("weather", "more_pairs_margin", margin, margin > 0)

### QueryMonth: the same pair in summer and in January

Expected before computing: "Zavala - Sv. Ante" (Hvar coast) and "K▲
(Hetvehely – Jakab-hegy – Cserkút)" (a 16 km Mecsek hill walk in southern
Hungary) are both warm (19–23 °C), dry and sunny in June–August, so with no
month they score fairly high (about 0.7). In January the coast stays mild
(about 8 °C) and the Pannonian hill is at freezing, so with
`QUERY_MONTH = 1` the score drops (about 0.5). Southness does not change with
the month.

In [ ]:
month_pair = pair_rows(["Zavala - Sv. Ante", "K▲ (Hetvehely – Jakab-hegy – Cserkút)"], [])
print(month_pair[[*climate_slice(None), *climate_slice(1), "southness"]].round(1).to_string())
for month in (None, 1):
    score = weather_score(month_pair.iloc[0], month_pair.iloc[1], month=month)
    print(f"weather_month\t{month}\t{score:.3f}")

### Weather synthetic checks

Same three checks, on the slice `QUERY_MONTH` selects. For reversed, the
Weather facts are rebuilt from the vertices in reverse order. The climate is a
mean along the line, and southness reads the same vertices in either
direction (counted from the nearer end), so the walking direction must not
matter.

In [ ]:
synthetic_checks(
    "weather",
    weather_score,
    weather_facts(check_dayhikes, check_vertices, dem_dir),
    weather_facts(check_dayhikes, backward_vertices, dem_dir),
)

## Water

Two measures: the capped number of WaterSources in the Corridor and the
DryStretch. The facts are built above (drinking water, springs, taps and
water points within 100 m of the line; each placed along the walk; the DryStretch is the
longest stretch between them, from the first vertex to the last).

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| `water_count`, capped at 3 | tolerance | cap 3, D = 3 | 58% of region DayHikes pass no WaterSource, 23% pass one, 9% two and 10% three or more (maximum 19). Past three more water adds nothing for a day hike, so the count is capped before it is compared. With D = 3, none against one scores 0.67, none against three 0; four points and nineteen are both "plenty" and score 1. A random pair's median is 0.67: 40% of random pairs share a level (mostly both none) and score 1, and most of the rest are one level apart. |
| `dry_stretch_m` | ratio | floor 1 km | A size, so a ratio: 3 km of dry walking against 6 km is as far apart as 6 against 12. Region middle half 2.3–6.4 km, median 3.9 km, 95th percentile 12.2 km, and a random pair's median is 0.49. The floor of 1 km treats any stretch under it as the same short hop; it binds on about 1% of region DayHikes (a village loop with a fountain every few hundred metres), so it hardly matters. |

The Water FacetScore (`water_score`) is the plain mean of the two. PRD
starting constants stay; the data does not argue for other values.

**What the data holds.** Springs count since the WaterSource re-ingest, and they
matter most in the mountains: the Water section above prints the split by highest point (55% of the
hikes above 1,500 m pass no WaterSource, against 73% when only drinking water
counted, on the Catalog before the main walk). Streams, troughs and unmapped
springs are still missing, so the Facet compares what OSM maps, not what is in
the ground. A DryStretch is also the hike's whole length for the 58% that pass
no WaterSource, so for those the Water Facet partly repeats Effort's length;
the count is the part that is new.

Result on the real region (below): the main similar pair scores 0.89 and the
dissimilar one 0.06. The four extra pairs separate with a margin of 0.74: the
lowest similar score (two dry karst walks, 0.95) against the highest
dissimilar (0.22, the hike whose only WaterSource is at its very end against a
well-watered town walk). Both soft targets are met.

In [ ]:
WATER_CAP = 3
WATER_COUNT_D = 3
DRY_STRETCH_FLOOR_M = 1_000


def capped_water_count(count):
    return None if pd.isna(count) else min(count, WATER_CAP)


def water_score(a, b):
    return facet_score(
        [
            tolerance(capped_water_count(a["water_count"]), capped_water_count(b["water_count"]), d=WATER_COUNT_D),
            ratio(a["dry_stretch_m"], b["dry_stretch_m"], floor=DRY_STRETCH_FLOOR_M),
        ]
    )


measure_spread(
    "water_count",
    lambda a, b: tolerance(capped_water_count(a), capped_water_count(b), d=WATER_COUNT_D),
    f"tolerance cap={WATER_CAP} D={WATER_COUNT_D}",
)
measure_spread("dry_stretch_m", lambda a, b: ratio(a, b, floor=DRY_STRETCH_FLOOR_M), f"ratio floor={DRY_STRETCH_FLOOR_M}")
print(f"water_count_share\t{facts['water_count'].clip(upper=WATER_CAP).value_counts(normalize=True).sort_index().round(3).to_dict()}")
print(f"dry_stretch_under_floor_share\t{(facts['dry_stretch_m'] < DRY_STRETCH_FLOOR_M).mean():.3f}")

### Water verification (real region)

Expected before computing:

- **Similar:** "Stara Fužina - Dom v Vojah" (Bohinj, 4 WaterSources) and
  "Kanjon Velika Paklenica" (Velebit, 8): well-watered 3.4 and 8.6 km walks.
  Both have the count at the cap (three or more) and a DryStretch of 1.9 and
  2.5 km (ratio 0.77), so they should score about 0.9. They are in different
  mountains, which Water ignores.
- **Dissimilar:** "Kanjon Velika Paklenica" against "Strogi rezervat prirode
  Bijele i Samarske stijene", a 19 km karst walk in Gorski kotar with no
  WaterSource at all: count 3 against 0 scores 0, and 2.5 km of dry stretch
  against 19.3 km about 0.13, so the Facet should be near 0.06.

The dry partner was "Put preko vrha Dinare" until springs were counted; it now
passes two springs on its 18 km ridge (DryStretch 10.9 km), so it is no longer a
hike without water.

In [ ]:
water_pairs = pair_rows(
    ["Stara Fužina - Dom v Vojah", "Kanjon Velika Paklenica"],
    ["Kanjon Velika Paklenica", "Strogi rezervat prirode Bijele i Samarske stijene"],
)
show_pairs(water_pairs, ["length_m", "water_count", "dry_stretch_m"], "Water pairs")
verify_pairs("water", water_score, water_pairs)

### More Water pairs (real region)

Expected before computing. Two that should be similar, two that should not:

- **Similar:** two dry karst walks, "Strogi rezervat prirode Bijele i Samarske
  stijene" (Gorski kotar, 19 km) and "Guge-Brezovac" (17.5 km up to the
  Brezovac hut on the Dinara karst above Knin): no WaterSource at all, so the
  count agrees and the DryStretch is the whole walk, 19.3 against 17.5 km
  (ratio 0.91, Facet about 0.95). And two short town walks, "Čez Ljubljanski
  grad" (7 points) and "Sentiero Tabor / Pešpot Tabor" (4): both are at the cap
  and below the 1 km floor, so they should score 1.
- **Dissimilar:** "Sentiero natura / Pešpot za Miramar", a Trieste seafront
  walk with 19 points, against "Wanderweg 338", a 19.5 km Styrian ridge walk
  with none (near 0). And "Lendweg" (7 points, 5 km walk, DryStretch 1.4 km)
  against "Šegova pot", a 14.7 km forest walk with one WaterSource at its very
  end, which cuts nothing: its DryStretch is its whole length. Count alone
  gives 0.33; the DryStretch (1.4 against 14.7 km, about 0.10) pulls the Facet
  down to about 0.2.

Two pairs were re-picked once springs were counted: "Put preko vrha Dinare"
now passes two springs, and "Z (Nemesvita – Büdös-kút – Vonyarcvashegy)" three
WaterSources (Büdös-kút is a spring), so neither plays its old part. Two more
were re-picked after the main walk (ADR-0012), whose walks left the Catalog as
over 20 km: "Kurtagić - Vaganj - Vrdovo" (now 25 km) and "Mala Kočevska
planinska pot" (now 53 km).

The check is the same as the main pair: the lowest similar score must be above
the highest dissimilar one.

In [ ]:
WATER_MORE = [
    ("similar", "Strogi rezervat prirode Bijele i Samarske stijene", "Guge-Brezovac"),
    ("similar", "Čez Ljubljanski grad", "Sentiero Tabor / Pešpot Tabor"),
    ("dissimilar", "Sentiero natura / Pešpot za Miramar", "Wanderweg 338"),
    ("dissimilar", "Lendweg", "Šegova pot"),
]

more_rows = pair_rows(sorted({n for _, a, b in WATER_MORE for n in (a, b)}), [])
more_table = pd.DataFrame(
    {
        "expected": expected,
        "a": a,
        "b": b,
        "counts": f"{int(more_rows.at[a, 'water_count'])} / {int(more_rows.at[b, 'water_count'])}",
        "dry_km": f"{more_rows.at[a, 'dry_stretch_m'] / 1000:.1f} / {more_rows.at[b, 'dry_stretch_m'] / 1000:.1f}",
        "score": water_score(more_rows.loc[a], more_rows.loc[b]),
    }
    for expected, a, b in WATER_MORE
)
print(more_table.round(3).to_string())
by_group = more_table.groupby("expected")["score"]
margin = by_group.min()["similar"] - by_group.max()["dissimilar"]
verify("water", "more_pairs_margin", margin, margin > 0)

### Water synthetic checks

Same three checks. For reversed, the Water facts are rebuilt from the vertices
in reverse order: the Corridor is measured to the line and each point is placed by
its closest place on it, so every position becomes length minus position and
the DryStretch stays the same.

In [ ]:
synthetic_checks(
    "water",
    water_score,
    water_facts(check_dayhikes, check_vertices, amenities),
    water_facts(check_dayhikes, backward_vertices, amenities),
)

## Facilities

One measure: presence over the five Facilities flags. The facts are built above:
a Hut, a restaurant or cafe and a Shelter within 100 m of the line, and a
restaurant or cafe and parking within 500 m of the Catalog Trailhead. The
Corridor is measured to the line, the Trailhead distance is a plain distance,
and a Shelter never counts as a Hut.

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| five flags (`hut_along`, `restaurant_cafe_along`, `shelter_along`, `restaurant_cafe_trailhead`, `parking_trailhead`) | presence | none | The share of the five flags on which the two DayHikes agree. Both-absent is agreement, so two hikes that both lack a Hut agree on that point. The flags are common enough to carry information (a Hut along 23% of region DayHikes, a restaurant or cafe along 34%, a Shelter 17%, a restaurant or cafe at the Trailhead 37%, parking 59%) and rare enough that most flags are false on both sides. |

The Facilities FacetScore (`facilities_score`) is that one number, so it comes
in fifths: 0, 0.2, 0.4, 0.6, 0.8 or 1. The flags are equally weighted; the data
gives no reason to favour one, and the Hiker's FacetWeights act on the Facet as
a whole.

**Presence is lenient.** Agreement on absence counts, so a Hut hike with no
other facility and a hike with nothing at all still agree on four flags out of
five and score 0.8. A random pair of region DayHikes scores a median 0.6. Read
the Facet by its extremes (all five alike against all five different) and not
by the dissimilar soft target alone, which a single missing Hut can miss
without the Facet being wrong. The same holds for the presence measures in
Landscape.

Result on the real region (below): the main similar pair scores 1.00 and the
dissimilar one 0.00; both soft targets are met. The four extra pairs separate
with a margin of 0.20: the lowest similar score is 1.00 (two hut-only hikes, two
lunch-stop walks) and the highest dissimilar is 0.80, the hut hike against the
no-facility hike, which differ on the Hut flag alone. The hut hike against the
lunch walk scores 0.20. The 0.4 soft target is therefore missed for the
one-flag case by design, not by error. A random pair's quartiles are 0.4, 0.6
and 0.8.

In [ ]:
FLAGS = list(FACILITIES_FLAG_COLUMNS)


def facilities_score(a, b):
    return facet_score([presence([a[c] for c in FLAGS], [b[c] for c in FLAGS])])


for flag in FLAGS:
    agree = np.mean([facts.at[a, flag] == facts.at[b, flag] for a, b in zip(left_ids, right_ids)])
    print(
        f"measure\t{flag}\tpresence\ttrue_share={facts[flag].mean():.3f}\t"
        f"missing={facts[flag].isna().mean():.3f}\trandom_pair_agree={agree:.2f}"
    )
pair_scores = [facilities_score(facts.loc[a], facts.loc[b]) for a, b in zip(left_ids, right_ids)]
q = np.quantile(pair_scores, [0.05, 0.25, 0.5, 0.75, 0.95]).round(2).tolist()
print(f"measure\tfacilities_score\tpresence\trandom_pair_q05,q25,q50,q75,q95={q}")
print("flags_set_share\t" + str(facts[FLAGS].sum(axis=1).value_counts(normalize=True).sort_index().round(3).to_dict()))

### Facilities verification (real region)

Expected before computing:

- **Similar:** "Cojzova koča - Skuta" and "Krekova koča - Altemaver - Kremant".
  Two Kamnik Alps hut hikes (to 2,490 m and 1,670 m): a Hut and a Shelter along
  the way, no restaurant or cafe, nothing at the Trailhead. The flags are
  identical, so the Facet should be 1. They are different routes of different
  length, which Facilities ignores.
- **Dissimilar:** "Šmartno - Koča pod Kremžarjevim vrhom" and "Draževnik -
  Gradišče". The first has all five flags (a Hut, a restaurant or cafe and a
  Shelter along the way, a restaurant or cafe and parking at the Trailhead); the
  second has none. They agree on nothing, so the Facet should be 0.

In [ ]:
facilities_pairs = pair_rows(
    ["Cojzova koča - Skuta", "Krekova koča - Altemaver - Kremant"],
    ["Šmartno - Koča pod Kremžarjevim vrhom", "Draževnik - Gradišče"],
)
show_pairs(facilities_pairs, ["length_m", *FLAGS], "Facilities pairs")
verify_pairs("facilities", facilities_score, facilities_pairs)

### More Facilities pairs (real region)

Expected before computing. Two that should be similar, two that should not:

- **Similar:** two hut-only hikes, "Okrešelj - Kamniško sedlo" and "Hudičev
  boršt - Kališče": a Hut along the way, nothing else, identical flags (1). And
  two walks with a lunch stop, "Kulinarische Weinberg-Wanderung" and "Wanderweg
  343": a restaurant or cafe along the way and a restaurant or cafe and parking
  at the Trailhead, no Hut and no Shelter. Identical flags (1).
- **Dissimilar:** the hut hike "Okrešelj - Kamniško sedlo" against the
  lunch walk "Kulinarische Weinberg-Wanderung" (a Hut and no meal against a meal
  and no Hut, four flags differ; about 0.2). And the hut hike against the
  no-facility hike "Mali Alan - Dušice" (a Hut and nothing else against nothing
  at all; only the Hut flag differs, so about 0.8). This second pair is the
  weak case named above: the Facet separates it from the similar pairs, but not
  by the 0.4 soft target.

The check is the same as the main pair: the lowest similar score must be above
the highest dissimilar one.

In [ ]:
FACILITIES_MORE = [
    ("similar", "Okrešelj - Kamniško sedlo", "Hudičev boršt - Kališče"),
    ("similar", "Kulinarische Weinberg-Wanderung", "Wanderweg 343"),
    ("dissimilar", "Okrešelj - Kamniško sedlo", "Kulinarische Weinberg-Wanderung"),
    ("dissimilar", "Okrešelj - Kamniško sedlo", "Mali Alan - Dušice"),
]

more_rows = pair_rows(sorted({n for _, a, b in FACILITIES_MORE for n in (a, b)}), [])
more_table = pd.DataFrame(
    {
        "expected": expected,
        "a": a,
        "b": b,
        "flags_a": "".join("1" if more_rows.at[a, c] else "0" for c in FLAGS),
        "flags_b": "".join("1" if more_rows.at[b, c] else "0" for c in FLAGS),
        "score": facilities_score(more_rows.loc[a], more_rows.loc[b]),
    }
    for expected, a, b in FACILITIES_MORE
)
print(more_table.round(3).to_string())
by_group = more_table.groupby("expected")["score"]
margin = by_group.min()["similar"] - by_group.max()["dissimilar"]
verify("facilities", "more_pairs_margin", margin, margin > 0)

### Facilities synthetic checks

Same three checks. For reversed, the Facilities facts are rebuilt from the
vertices in reverse order: the Corridor is measured to the line and the Trailhead
distance does not use the line at all, so the walking direction must not matter.

In [ ]:
synthetic_checks(
    "facilities",
    facilities_score,
    facilities_facts(check_dayhikes, check_vertices, amenities),
    facilities_facts(check_dayhikes, backward_vertices, amenities),
)

## Trail

Two measures: the SurfaceMix and the TarmacApproach. The facts are built above
(four length fractions over the classified walked member ways, from the
`surface` tag else the highway kind; the paved length from the Trailhead
until the first non-paved way).

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| SurfaceMix (four shares) | mix | none | Scores the shared share: all natural path against half natural, half gravel is 0.5. A random pair of region DayHikes shares a median 0.53 (natural is half of the average hike, gravel 27%, paved 22%, rock 1%). |
| `tarmac_approach_m` | tolerance | D = 2 km | 57% of region DayHikes have no approach at all (the first way is unpaved); the 75th percentile is 0.43 km, the 90th 1.4 km, the 95th 2.4 km and the maximum 12.2 km. D = 2 km scores a difference of 2 km or more as 0, and 0.5 km of road against none as 0.75. A random pair's median is 0.88 because most pairs both have none. |

The Trail FacetScore (`trail_score`) is the plain mean of the two. The PRD
starting constant (D = 2 km) stays; the data does not argue for another (a
median random pair would score 0.76 at D = 1 km and 0.95 at D = 5 km, and the
95th percentile is 2.4 km).

Scope of the mix: 47% of walked ways carry a listed `surface` tag; the other 53%
(tracks and paths mostly, then residential and unclassified roads) come from
the highway kind, which is why the mix stays defined for nearly every hike.
Mixes are over the classified length, so a hike's unclassified bits (46 ways
in the region) do not count against it.

A loop starts and ends at the Trailhead, so either direction is a fair
start: its TarmacApproach is the shorter of the two.

Result on the real region (below): the main similar pair scores 1.00 and the
dissimilar one 0.00 (a forest path against a promenade share nothing, and
differ by 3.6 km of road). The four extra pairs separate with a margin of 0.67:
the lowest similar score (the two road-approach hikes, 0.71) against the
highest dissimilar (0.04, the gravel track against a paved walk).

In [ ]:
TARMAC_D_M = 2_000
SURFACE = list(SURFACE_SHARE_COLUMNS)


def trail_score(a, b):
    return facet_score(
        [
            mix([a[c] for c in SURFACE], [b[c] for c in SURFACE]),
            tolerance(a["tarmac_approach_m"], b["tarmac_approach_m"], d=TARMAC_D_M),
        ]
    )


mix_scores = [
    mix([facts.at[a, c] for c in SURFACE], [facts.at[b, c] for c in SURFACE])
    for a, b in zip(left_ids, right_ids)
]
mix_scores = [s for s in mix_scores if s is not None]
q = np.quantile(mix_scores, [0.05, 0.25, 0.5, 0.75, 0.95]).round(2).tolist()
print(
    f"measure\tsurface_mix\tmix\tmissing={facts[SURFACE].isna().any(axis=1).mean():.3f}\t"
    f"random_pair_q05,q25,q50,q75,q95={q}"
)
for column in SURFACE:
    print(f"share\t{column}\tmean={facts[column].mean():.3f}\tq95={facts[column].quantile(0.95):.3f}")
print(f"tarmac_zero_share\t{(facts['tarmac_approach_m'] == 0).mean():.3f}")
measure_spread("tarmac_approach_m", lambda a, b: tolerance(a, b, d=TARMAC_D_M), f"tolerance D={TARMAC_D_M}")

### Trail verification (real region)

Expected before computing:

- **Similar:** "Dom Paklenica - Jurline" and "Stap - Jelova Ruja". Both are
  all-natural forest paths in the Velebit (0 m of road before the trail), so
  the mix is the same and the approach is the same.
- **Dissimilar:** "Krč - Lišćani bunar" and "314 Cres – Lungomare". The first
  is a forest path with no road; the second is a seaside promenade, paved from
  end to end, with 3.6 km of tarmac approach.

In [ ]:
trail_pairs = pair_rows(
    ["Dom Paklenica - Jurline", "Stap - Jelova Ruja"],
    ["Krč - Lišćani bunar", "314 Cres – Lungomare"],
)
show_pairs(trail_pairs, [*SURFACE, "tarmac_approach_m"], "Trail pairs")
verify_pairs("trail", trail_score, trail_pairs)

### More Trail pairs (real region)

Expected before computing. Two that should be similar, two that should not:

- **Similar:** "Planinarska staza Medvednica 18" and "Planinarska staza
  Medvednica 36": two gravel tracks on the same mountain (73% and all gravel),
  with at most 0.3 km of road approach. "Jelovec - Žavcarjev vrh" and
  "Bresternica - Žavcarjev vrh": both leave a Pohorje village by road (3.6 and
  4.5 km of tarmac) and finish on forest tracks, half paved.
- **Dissimilar:** the forest path "Dom Paklenica - Jurline" against the
  asphalt "135 ~ Pot kurirjev in vezistov NOV Slovenije" (11.6 km of road, all
  paved), and the gravel "Planinarska staza Medvednica 18" against the paved
  "Stepinčev put 2. etapa" (5.4 km of road approach).

The all-paved walk was "Welschlauf Südsteiermark" (13.6 km of road) until the
main walk (ADR-0012) made it a 36 km walk, which left the Catalog.

The check is the same as the main pair: the lowest similar score must be above
the highest dissimilar one.

In [ ]:
TRAIL_MORE = [
    ("similar", "Planinarska staza Medvednica 18", "Planinarska staza Medvednica 36"),
    ("similar", "Jelovec - Žavcarjev vrh", "Bresternica - Žavcarjev vrh"),
    ("dissimilar", "Dom Paklenica - Jurline", "135 ~ Pot kurirjev in vezistov NOV Slovenije"),
    ("dissimilar", "Planinarska staza Medvednica 18", "Stepinčev put 2. etapa"),
]

more_rows = pair_rows(sorted({n for _, a, b in TRAIL_MORE for n in (a, b)}), [])
more_table = pd.DataFrame(
    {
        "expected": expected,
        "a": a,
        "b": b,
        "surface_mix": mix([more_rows.at[a, c] for c in SURFACE], [more_rows.at[b, c] for c in SURFACE]),
        "tarmac": tolerance(more_rows.at[a, "tarmac_approach_m"], more_rows.at[b, "tarmac_approach_m"], d=TARMAC_D_M),
        "score": trail_score(more_rows.loc[a], more_rows.loc[b]),
    }
    for expected, a, b in TRAIL_MORE
)
print(more_table.round(3).to_string())
by_group = more_table.groupby("expected")["score"]
margin = by_group.min()["similar"] - by_group.max()["dissimilar"]
verify("trail", "more_pairs_margin", margin, margin > 0)

### Trail synthetic checks

Same three checks. For reversed, the Trail facts are rebuilt from the vertices
in reverse order: the ways are taken in their stored walk order, which starts
at the Trailhead, and a loop counts the shorter of its two directions, so the
walking direction must not matter.

In [ ]:
synthetic_checks(
    "trail",
    trail_score,
    trail_facts(check_dayhikes, check_vertices, check_ways),
    trail_facts(check_dayhikes, backward_vertices, check_ways),
)

## RoundTrip

One measure: presence on the one `round_trip` flag. The fact is built above:
the main walk ends within 50 m (`GAP_JOIN_M`) of its Trailhead. A
lollipop (stem plus loop) and a line are not RoundTrips.

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| `round_trip` | presence | none | The question is yes or no: is `length_m` the whole walk back to the car? Two RoundTrips agree (1), a RoundTrip and a line do not (0). Two lines agree too: both need a second leg, walked back or by shuttle. 8.4% of region DayHikes are RoundTrips, so a random pair agrees 87% of the time, mostly on two lines. |

The RoundTrip FacetScore (`round_trip_score`) is that one number, so it is
0 or 1. A Seed's RoundTrip matters most when the Seed is one: with weight on
this Facet the Ranking prefers the 8% of hikes that also close.

Result on the real region (below): the main similar pair scores 1 and the
dissimilar one 0; both soft targets are met. The extra pairs separate with a
margin of 1: the lollipop agrees with the line and not with the RoundTrip.

In [ ]:
def round_trip_score(a, b):
    return facet_score([presence([a["round_trip"]], [b["round_trip"]])])


agree = np.mean([facts.at[a, "round_trip"] == facts.at[b, "round_trip"] for a, b in zip(left_ids, right_ids)])
print(
    f"measure\tround_trip\tpresence\ttrue_share={facts['round_trip'].mean():.3f}\t"
    f"missing={facts['round_trip'].isna().mean():.3f}\trandom_pair_agree={agree:.2f}"
)

### RoundTrip verification (real region)

Expected before computing:

- **Similar:** "Po stopinjah pastirjev" and "Anello della Pecorara". Both close
  where they start: a 6.4 km round on the Velika planina plateau (to 1,660 m)
  and a 5.9 km round near the coast (to 180 m). Nothing else about them is
  alike, which RoundTrip ignores, so the Facet should be 1.
- **Dissimilar:** "Po stopinjah pastirjev" and "Velika planina - Planina Dol".
  Same plateau, Trailheads 2 km apart, but the second is a 4.1 km line whose
  ends are 3.5 km apart. The Facet should be 0.

In [ ]:
round_trip_pairs = pair_rows(
    ["Po stopinjah pastirjev", "Anello della Pecorara"],
    ["Po stopinjah pastirjev", "Velika planina - Planina Dol"],
)
show_pairs(round_trip_pairs.iloc[[0, 1, 3]], ["length_m", "highest_m", "round_trip"], "RoundTrip pairs")
verify_pairs("round_trip", round_trip_score, round_trip_pairs)

### More RoundTrip pairs (real region)

Expected before computing. The lollipop is the case to check:

- **Similar:** two RoundTrips in different countries, "Razgledni vrhovi nad
  Črnim Vrhom" (Slovenia) and "Verborgene Gletscherwelt Runde" (Austria) (1).
  And the lollipop "Arriacher Hofwanderweg-Variante" against the line "Velika
  planina - Planina Dol": the lollipop's walk ends about 280 m from its
  Trailhead, where its stem meets the loop, so like the line it needs walking
  back (1).
- **Dissimilar:** the lollipop "Arriacher Hofwanderweg-Variante" against the
  RoundTrip "Verborgene Gletscherwelt Runde" (0). Both are loops on the map,
  but only one ends at the car.

The check is the same as the main pair: the lowest similar score must be above
the highest dissimilar one.

In [ ]:
ROUND_TRIP_MORE = [
    ("similar", "Razgledni vrhovi nad Črnim Vrhom", "Verborgene Gletscherwelt Runde"),
    ("similar", "Arriacher Hofwanderweg-Variante", "Velika planina - Planina Dol"),
    ("dissimilar", "Arriacher Hofwanderweg-Variante", "Verborgene Gletscherwelt Runde"),
]

more_rows = pair_rows(sorted({n for _, a, b in ROUND_TRIP_MORE for n in (a, b)}), [])
more_table = pd.DataFrame(
    {
        "expected": expected,
        "a": a,
        "b": b,
        "round_trip_a": more_rows.at[a, "round_trip"],
        "round_trip_b": more_rows.at[b, "round_trip"],
        "score": round_trip_score(more_rows.loc[a], more_rows.loc[b]),
    }
    for expected, a, b in ROUND_TRIP_MORE
)
print(more_table.round(3).to_string())
by_group = more_table.groupby("expected")["score"]
margin = by_group.min()["similar"] - by_group.max()["dissimilar"]
verify("round_trip", "more_pairs_margin", margin, margin > 0)

### RoundTrip synthetic checks

Same three checks. For reversed, the fact is rebuilt from the vertices in
reverse order: the distance between the two ends does not depend on which one
comes first.

In [ ]:
synthetic_checks(
    "round_trip",
    round_trip_score,
    round_trip_facts(check_dayhikes, check_vertices),
    round_trip_facts(check_dayhikes, backward_vertices),
)

## Steepness

Two measures: ClimbGrade and SteepestKm. The facts are built above:
elevation resampled every ~50 m along the main walk; ClimbGrade is the gain
over the length of the climbing stretches (in the direction with the larger
climb), SteepestKm the largest height change, up or down, over any 1 km.

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| `climb_grade` | ratio | floor 0.03 | A grade is a size: 10 % vs 20 % is "half as steep", as 300 m vs 600 m of gain is half the climb. The middle half of the region climbs at 8–17 %. About 5 % climb at under 3 %: flat walks, which should match each other (1 % vs 2 % scores 1), not sit at the far end of a ratio. |
| `steepest_km_m` | ratio | floor 50 m | The middle half of the region has a steepest kilometre of 90–225 m. Under 50 m (about one region DayHike in ten) the walk has no real climb, so those match each other. |

The FacetScore (`steepness_score`) is the mean of the two. Effort compares the
totals (length, gain); Steepness compares how hard the climbing is, so a
short steep line and a long steep RoundTrip score alike here and unlike on
Effort.

Result on the real region (below): 4.7% of DayHikes climb at under 3 % and
10.8% have a steepest kilometre under 50 m. A random pair scores a median 0.55
on ClimbGrade and 0.53 on SteepestKm. Both floors are kept.

In [ ]:
CLIMB_GRADE_FLOOR = 0.03
STEEPEST_KM_FLOOR_M = 50


def steepness_score(a, b):
    return facet_score(
        [
            ratio(a["climb_grade"], b["climb_grade"], floor=CLIMB_GRADE_FLOOR),
            ratio(a["steepest_km_m"], b["steepest_km_m"], floor=STEEPEST_KM_FLOOR_M),
        ]
    )


measure_spread("climb_grade", lambda a, b: ratio(a, b, floor=CLIMB_GRADE_FLOOR), f"ratio floor={CLIMB_GRADE_FLOOR}")
measure_spread("steepest_km_m", lambda a, b: ratio(a, b, floor=STEEPEST_KM_FLOOR_M), f"ratio floor={STEEPEST_KM_FLOOR_M}")
print(f"climb_grade_under_floor\t{(facts['climb_grade'] < CLIMB_GRADE_FLOOR).mean():.3f}")
print(f"steepest_km_under_floor\t{(facts['steepest_km_m'] < STEEPEST_KM_FLOOR_M).mean():.3f}")

### Steepness verification (real region)

Expected before computing:

- **Similar:** "Kalanova pot" and "Uniški batovi". The first is a 2.9 km line
  with 470 m of gain; the second a 7.1 km RoundTrip with 815 m to 1,700 m.
  Effort sees them apart (length ratio 0.41). Both climb at about 20 % with a
  steepest kilometre of about 300 m, so Steepness should be high.
- **Dissimilar:** "Uniški batovi" and "Hüttenwanderweg". Both are RoundTrips
  of 7–8 km that top out at about 1,700 m, so Altitude sees them close. The
  Hüttenwanderweg strolls between huts on an alpine slope: about 340 m of gain,
  climbing at about 7 % with no kilometre rising more than about 90 m.
  Steepness should be low.

Result on the real region (below): the similar pair scores 0.952 and the
dissimilar one 0.321; both soft targets are met. The other Facets show why
Steepness is its own Facet: the similar pair scores only 0.495 on Effort and
0.323 on Altitude, the dissimilar pair 0.543 on Effort and 0.731 on Altitude.

In [ ]:
steepness_pairs = pair_rows(
    ["Kalanova pot", "Uniški batovi"],
    ["Uniški batovi", "Hüttenwanderweg"],
)
show_pairs(
    steepness_pairs.iloc[[0, 1, 3]],
    ["length_m", "elevation_gain_m", "highest_m", "round_trip", "climb_grade", "steepest_km_m"],
    "Steepness pairs",
)
verify_pairs("steepness", steepness_score, steepness_pairs)
for pair, (a, b) in (("similar", (0, 1)), ("dissimilar", (2, 3))):
    print(
        f"other_facets\t{pair}\teffort={effort_score(steepness_pairs.iloc[a], steepness_pairs.iloc[b]):.3f}"
        f"\taltitude={altitude_score(steepness_pairs.iloc[a], steepness_pairs.iloc[b]):.3f}"
    )

### Steepness synthetic checks

Same three checks. For reversed, the facts are rebuilt from the vertices in
reverse order: the climbing direction is picked by the larger climb and
SteepestKm takes the absolute height change, so neither depends on which end
comes first.

In [ ]:
synthetic_checks(
    "steepness",
    steepness_score,
    steepness_facts(check_dayhikes, check_vertices),
    steepness_facts(check_dayhikes, backward_vertices),
)

## Combined score: one Seed, two FacetWeights sets

Every region DayHike gets all ten FacetScores against one Seed, computed
once. FacetWeights only change how they combine (`combined_score`: Σ wᶠ·sᶠ / Σ wᶠ
over the Facets that are scored and have wᶠ > 0, plus coverage). The Seed is
"Aljažev dom - Kredarica - Triglav": a 5.8 km,
2,230 m climb from 1,000 m past the Kredarica hut to Triglav at 2,800 m, with
a Hut and a Shelter along the way and parking at the Trailhead, and no
restaurant or cafe.

| Facet | Set A `facilities_heavy` | Set B `altitude_heavy` |
| --- | --- | --- |
| Effort | 0 | 20 |
| Altitude | 30 | 60 |
| Profile | 0 | 0 |
| Landscape | 20 | 20 |
| Weather | 0 | 0 |
| Water | 0 | 0 |
| Facilities | 50 | 0 |
| Trail | 0 | 0 |
| RoundTrip | 0 | 0 |
| Steepness | 0 | 0 |

Set A is the PRD example: "I care about huts and how high it goes, not about
the trees." Set B makes Altitude the main theme. It keeps Landscape at 20 and
gives Facilities' weight to Effort, so a hike that climbs as high from as high,
with the same work, wins whatever its Huts.

The tables show all ten FacetScores, including the ones a set weights 0, so
you can see what each set ignores. `weather_score` uses the `QUERY_MONTH` that
was set when the Weather cell ran. After you change it, re-run that cell and
then this one.

In [ ]:
FACET_SCORES = {
    "effort": effort_score,
    "altitude": altitude_score,
    "profile": profile_score,
    "landscape": landscape_score,
    "weather": weather_score,  # binds QUERY_MONTH when the Weather cell ran
    "water": water_score,
    "facilities": facilities_score,
    "trail": trail_score,
    "round_trip": round_trip_score,
    "steepness": steepness_score,
}
FACET_WEIGHTS = {
    "facilities_heavy": {
        "effort": 0, "altitude": 30, "profile": 0, "landscape": 20,
        "weather": 0, "water": 0, "facilities": 50, "trail": 0,
        "round_trip": 0, "steepness": 0,
    },
    "altitude_heavy": {
        "effort": 20, "altitude": 60, "profile": 0, "landscape": 20,
        "weather": 0, "water": 0, "facilities": 0, "trail": 0,
        "round_trip": 0, "steepness": 0,
    },
}
SEED = "Aljažev dom - Kredarica - Triglav"


def seed_row(name):
    rows = facts[facts["name"] == name]
    assert len(rows) == 1, f"Seed name must be unique: {name}"
    return rows.iloc[0]


def facet_scores_against(seed, candidates):
    """All ten FacetScores of each candidate against one Seed (NaN = missing)."""
    scores = {
        dayhike_id: {facet: fn(seed, row) for facet, fn in FACET_SCORES.items()}
        for dayhike_id, row in candidates.iterrows()
    }
    return pd.DataFrame.from_dict(scores, orient="index", columns=list(FACET_SCORES)).astype(float)


def rank(scores, weights):
    """Combined score and coverage per candidate, best first."""
    combined = [combined_score(row.to_dict(), weights) for _, row in scores.iterrows()]
    ranking = scores.assign(
        score=[np.nan if s is None else s for s, _ in combined],
        coverage=[c for _, c in combined],
    )
    ranking.insert(0, "name", facts.loc[scores.index, "name"])
    return ranking.sort_values("score", ascending=False)


seed = seed_row(SEED)
seed_scores = facet_scores_against(seed, facts.drop(index=seed.name))
print(f"seed\t{SEED}\tquery_month\t{QUERY_MONTH}")
print(
    facts.loc[[seed.name], ["length_m", "elevation_gain_m", "highest_m", "lowest_m", *FLAGS, "tree_share", "bare_share"]]
    .round(2)
    .T.to_string()
)
rankings = {}
for set_name, weights in FACET_WEIGHTS.items():
    rankings[set_name] = rank(seed_scores, weights)
    top = rankings[set_name].head(5)
    print(f"\n{set_name}: " + ", ".join(f"{facet} {w}" for facet, w in weights.items() if w > 0))
    print(top.round(3).to_string())
    for _, row in top.iterrows():
        print(f"rank\t{set_name}\t{row['name']}\t{row['score']:.3f}\t{row['coverage']:.2f}")
shared = set(rankings["facilities_heavy"].head(5).index) & set(rankings["altitude_heavy"].head(5).index)
print(f"top5_shared\t{len(shared)}")

In [ ]:
def show_ranking_map(seed, ranking, title):
    """The Seed (thick orange) and the top five (blue, numbered by rank) on one map."""
    top = gpd.GeoDataFrame(facts.loc[ranking.head(5).index], geometry="geometry", crs=facts.crs).to_crs(epsg=3857)
    seed_line = gpd.GeoDataFrame([seed], geometry="geometry", crs=facts.crs).to_crs(epsg=3857)
    fig, ax = plt.subplots(figsize=(7, 7))
    top.plot(ax=ax, color="#1d4ed8", linewidth=2)
    seed_line.plot(ax=ax, color="#c2410c", linewidth=4)
    for number, (_, row) in enumerate(top.iterrows(), start=1):
        point = row.geometry.representative_point()
        ax.annotate(f"{number}. {row['name']}", (point.x, point.y), fontsize=7, color="#1d4ed8")
    point = seed_line.geometry.iloc[0].representative_point()
    ax.annotate(f"Seed: {seed['name']}", (point.x, point.y), fontsize=8, color="#c2410c", weight="bold")
    try:
        import contextily as cx

        cx.add_basemap(ax, source=cx.providers.OpenTopoMap)
    except Exception as error:  # offline: the lines alone are still the map
        print("basemap skipped", type(error).__name__)
    ax.set_title(title, fontsize=9)
    ax.set_axis_off()
    fig.tight_layout()
    show_fig(fig)


for set_name, weights in FACET_WEIGHTS.items():
    show_ranking_map(
        seed,
        rankings[set_name],
        f"{set_name}: " + ", ".join(f"{facet} {w}" for facet, w in weights.items() if w > 0),
    )

### Why the two top fives differ

Result on the real region (above): the two top fives share two DayHikes,
"Aljažev dom - Škrlatica" (1st under both) and "Hanzova pot" (4th under A, 3rd
under B). "Aljažev dom - Škrlatica" also leaves from Aljažev dom (999 m), climbs
2,240 m over 6.0 km (Effort 0.76) to 2,650 m (Altitude 0.92) and has the same
five flags as the Seed (Facilities 1.0). The other places follow the weights:

- **Set A puts half its weight on Facilities.** The Seed has a Hut and a
  Shelter along the way and parking at the Trailhead, and no restaurant or
  cafe. Only 1.3% of region DayHikes have exactly those five flags
  (Facilities 1.0). Two of set A's top five are among them; the other three
  differ on one flag (0.8). Effort weighs 0 in A, so "625" ranks second: it is
  11.7 km long against the Seed's 5.8 km (Effort 0.61).
- **Set B puts 60 on Altitude and 20 on Effort.** "Tamar - Kotovo sedlo -
  Jalovec" is second under B. It goes from 1,110 to 2,580 m (Altitude 0.84)
  and climbs 2,000 m over 6.1 km (Effort 0.84). It has no parking at the
  Trailhead, so its Facilities score is 0.8, and under A it ranks 7th. "103"
  goes from 880 to 2,720 m (Altitude 0.90) but agrees with the Seed on only
  two of the five flags (Facilities 0.4); it is 4th under B and 295th under A.
- **It works the other way too.** "625" is 2nd under A, with matching flags,
  but it tops out at 2,230 m against the Seed's 2,800 m (Altitude 0.71).
  Under B it drops to 23rd.

Coverage is 1.00 for every candidate in both sets: no Facet is missing in this
region (see Coverage below).

### Weight on RoundTrip

Both sets above weigh RoundTrip 0, so their rankings do not change with it.
This cell takes a Seed that is a RoundTrip, "Po stopinjah pastirjev", and ranks under set B twice: as it is,
and with RoundTrip weighted 50 on top. It prints how many of each top five
share the Seed's RoundTrip value. Without the weight the top five follow
Altitude and Effort whatever their ends; with it, hikes that also close where
they start move up.

Result on the real region (below): under set B alone none of the top five is a
RoundTrip. Its top three are a synthetic-label relation, a Montenegrin trail
and "Šantić-Ljuta Greda", all lines (RoundTrip 0) that match Altitude (0.93 to
0.98) and Effort. With RoundTrip at 50 all five are RoundTrips, led by
"Sentiero Naturalistico del Matajur" (0.900: Effort 0.90, Altitude 0.91).

In [ ]:
ROUND_TRIP_SEED = "Po stopinjah pastirjev"
round_trip_seed = seed_row(ROUND_TRIP_SEED)
round_trip_seed_scores = facet_scores_against(round_trip_seed, facts.drop(index=round_trip_seed.name))
print(f"seed\t{ROUND_TRIP_SEED}\tround_trip\t{bool(round_trip_seed['round_trip'])}")
for set_name, weights in (
    ("altitude_heavy", FACET_WEIGHTS["altitude_heavy"]),
    ("altitude_heavy_round_trip", {**FACET_WEIGHTS["altitude_heavy"], "round_trip": 50}),
):
    top = rank(round_trip_seed_scores, weights).head(5)
    same = int((facts.loc[top.index, "round_trip"] == round_trip_seed["round_trip"]).sum())
    print(f"\n{set_name}: " + ", ".join(f"{facet} {w}" for facet, w in weights.items() if w > 0))
    print(top.assign(is_round_trip=facts.loc[top.index, "round_trip"]).round(3).to_string())
    print(f"round_trip_top5_same\t{set_name}\t{same}\tof\t5")

### Weight on Steepness

Both sets weigh Steepness 0. This cell takes a steep Seed, "Uniški batovi"
(7.1 km at about 20 %), and ranks under set B
twice: as it is, and with Steepness weighted 50 on top. It prints each top
five's ClimbGrade and the median gap to the Seed's. Set B matches the Seed's
height and totals; with weight on Steepness the top five also climb as
steeply.

Result on the real region (below): under set B alone the top five climb at
10–23 %, a median 0.062 off the Seed's 0.205, led by "134" (14 %,
Steepness 0.70). With Steepness at 50 all five climb at 19.5–23 %, a median
0.005 off, led by "663 Rjautza" (20 %, Steepness 0.92). Two of the five stay.

In [ ]:
STEEP_SEED = "Uniški batovi"
steep_seed = seed_row(STEEP_SEED)
steep_seed_scores = facet_scores_against(steep_seed, facts.drop(index=steep_seed.name))
print(f"seed\t{STEEP_SEED}\tclimb_grade\t{steep_seed['climb_grade']:.3f}\tsteepest_km_m\t{steep_seed['steepest_km_m']:.0f}")
for set_name, weights in (
    ("altitude_heavy", FACET_WEIGHTS["altitude_heavy"]),
    ("altitude_heavy_steepness", {**FACET_WEIGHTS["altitude_heavy"], "steepness": 50}),
):
    top = rank(steep_seed_scores, weights).head(5)
    grades = facts.loc[top.index, "climb_grade"]
    print(f"\n{set_name}: " + ", ".join(f"{facet} {w}" for facet, w in weights.items() if w > 0))
    print(top.assign(climb_grade=grades, steepest_km_m=facts.loc[top.index, "steepest_km_m"]).round(3).to_string())
    print(f"steepness_top5_grade_gap\t{set_name}\t{(grades - steep_seed['climb_grade']).abs().median():.3f}")

## Several Seeds: the maximum rule

With several Seeds, a candidate's score is its best combined score against any
one Seed (`multi_seed_score`, as ADR-0003 does for production Ranking), never
the mean. Two dissimilar Seeds are used here: the alpine "Aljažev dom -
Kredarica - Triglav" and the Hvar coast climb "Zavala - Sv. Ante", both with set B weights. Every
candidate is scored against each Seed. Each `multi_seed` line prints the score
followed by the two per-Seed scores, so you can check by eye that the score is
the larger one; the cell also asserts this for every candidate. The `mean`
column shows what averaging would give instead.

Result on the real region (below): the coastal Seed wins 3,229 candidates and
the alpine Seed 1,353. The top ten are all low walks that score under 0.15
against Kredarica ("Francuska cesta": 0.904 against Zavala, 0.135 against
Kredarica), because the region has many more DayHikes like Zavala than like
Kredarica. The first candidate the alpine Seed wins ranks 20th: "Aljažev dom -
Škrlatica" at 0.838, the same score it gets against Kredarica alone. Each
score belongs to one Seed and is never stitched from both. Averaging would rank
hikes by how little they resemble either Seed: every top mean is about 0.5, and
"Kamenjak - Premužićeva staza" (0.23 / 0.82) heads that list.

In [ ]:
SEEDS = ["Aljažev dom - Kredarica - Triglav", "Zavala - Sv. Ante"]
MULTI_WEIGHTS = FACET_WEIGHTS["altitude_heavy"]

seeds = [seed_row(name) for name in SEEDS]
candidates = facts.drop(index=[s.name for s in seeds])
multi = pd.DataFrame(
    {"name": candidates["name"], **{s["name"]: rank(facet_scores_against(s, candidates), MULTI_WEIGHTS)["score"] for s in seeds}}
)
multi["score"] = pd.Series(
    [multi_seed_score([a, b]) for a, b in zip(multi[SEEDS[0]], multi[SEEDS[1]])], index=multi.index, dtype=float
)
multi["mean"] = multi[SEEDS].mean(axis=1)
multi = multi.sort_values("score", ascending=False)
assert np.allclose(multi["score"], multi[SEEDS].max(axis=1), equal_nan=True), "score must be the higher of the two"

print("seeds", SEEDS, "weights", {facet: w for facet, w in MULTI_WEIGHTS.items() if w > 0})
print(multi.head(10).round(3).to_string())
for _, row in multi.head(10).iterrows():
    print(f"multi_seed\t{row['name']}\t{row['score']:.3f}\t{row[SEEDS[0]]:.3f}\t{row[SEEDS[1]]:.3f}")
won_by_first = (multi[SEEDS[0]] >= multi[SEEDS[1]]).to_numpy()
for name, won in ((SEEDS[0], won_by_first), (SEEDS[1], ~won_by_first)):
    position = int(np.argmax(won)) + 1 if won.any() else None
    print(f"multi_seed_won\t{name}\tcandidates={int(won.sum())}\tbest_rank={position}")
by_mean = multi.sort_values("mean", ascending=False).head(5)
print("top 5 if the two scores were averaged instead:")
print(by_mean.round(3).to_string())

## Coverage: how often each Facet is missing

A Facet is missing for a DayHike when the DayHike cannot be scored on it even
against itself (every measure missing). The Facet then drops out of the
combined score, and coverage falls below 1. The cell prints that share for each
Facet (`facet_missing`), then every fact that is ever missing, then the share of
ranked candidates whose coverage is below 1 under each FacetWeights set.

Result on the real region (below): no Facet is missing for any of the 4,584
DayHikes, so coverage is 1.00 in both rankings. Two facts are missing inside a
Facet and drop out of its mean:

- Difficulty, on 45.8% of DayHikes. Effort then rests on length and gain.
- Southness, on 1.0% (44 flat lowland walks with no point steeper than 5°).
  Weather then rests on the three climate values.

Climate is never missing, for the regime or for any month. The Facilities flags,
the Landscape presence facts and RoundTrip are never missing by construction (no
amenity in range means False). Coverage will matter outside the sampled
rasters, in the whole-of-Europe phase, but not in this region.

In [ ]:
MEASURE_COLUMNS = {
    "effort": ["length_m", "elevation_gain_m", "difficulty_rank"],
    "altitude": ["highest_m", "lowest_m"],
    "profile": ["profile_shape"],
    "landscape": [*LANDCOVER, "peak_summited", "viewpoint_along", "peaks_around"],
    "weather": [*climate_slice(QUERY_MONTH), "southness"],
    "water": ["water_count", "dry_stretch_m"],
    "facilities": FLAGS,
    "trail": [*SURFACE, "tarmac_approach_m"],
    "round_trip": ["round_trip"],
    "steepness": ["climb_grade", "steepest_km_m"],
}

# A Facet is missing for a DayHike when it cannot be scored even against itself.
unscored = {facet: 0 for facet in FACET_SCORES}
for _, row in facts.iterrows():
    for facet, fn in FACET_SCORES.items():
        unscored[facet] += fn(row, row) is None
for facet, count in unscored.items():
    print(f"facet_missing\t{facet}\t{count / len(facts):.3f}")

measure_missing = pd.DataFrame(
    [
        {"facet": facet, "fact": column, "missing": facts[column].isna().mean()}
        for facet, columns in MEASURE_COLUMNS.items()
        for column in columns
    ]
)
print(measure_missing[measure_missing["missing"] > 0].round(3).to_string(index=False))
print(f"facts_never_missing\t{(measure_missing['missing'] == 0).sum()} of {len(measure_missing)}")
for set_name, ranking in rankings.items():
    print(f"coverage_below_1\t{set_name}\t{(ranking['coverage'] < 1).mean():.3f}")

## Verification summary

This table collects every `verify` line above (Facet, check, score,
PASS/FAIL). `verify` records each line as it prints it. Below the table: the
count of passes and failures, each failure, and each missed soft target (from
the `soft` lines).

In [ ]:
summary = pd.DataFrame(
    [{"facet": facet, "check": check, "score": score, "result": result} for (facet, check), (score, result) in VERIFY.items()]
)
print(summary.round(3).to_string(index=False))
passes = int((summary["result"] == "PASS").sum())
print(f"verify_summary\tpass={passes}\tfail={len(summary) - passes}")
for row in summary[summary["result"] == "FAIL"].itertuples():
    print(f"verify_failed\t{row.facet}\t{row.check}\t{row.score:.3f}")
for (facet, check), (score, target, met) in SOFT.items():
    if not met:
        print(f"soft_missed\t{facet}\t{check}\t{score:.3f}\t{target}")

### Reading the summary (real region)

56 checks: 56 PASS, 0 FAIL. They are 20 named-pair lines (a similar and a
dissimilar pair for each of the ten Facets), 6 more-pairs margins (Landscape,
Weather, Water, Facilities, Trail, RoundTrip) and 30 synthetic checks (identity,
symmetry, reversed for each Facet).

**Failures:** none.

**Missed soft targets.** These are known design points, not failures. Each
pair still scores below its similar pair:

- **Profile, dissimilar pair 0.702** (target ≤ 0.4). Two ProfileShapes that both
  go up and down share much of the unit square: a random pair's median is
  0.77, and even flat against a straight climb scores 0.52. The verdict rests on
  similar > dissimilar (0.984 against 0.702).
- **Landscape, forest against open karst 0.58** ("Predmeja - Iztokova koča"
  against "Zahum Idovac", in the more-pairs table). The LandCover mixes share
  only 0.04, but neither hike has a summit or a Viewpoint. Both-absent flags
  agree, so 2 of the 4 measures score 1.
- **Facilities, one-flag difference 0.8** ("Okrešelj - Kamniško sedlo" against
  "Mali Alan - Dušice", in the more-pairs table). Presence is lenient: a Hut
  hike with no other facility and a hike with none agree on 4 of 5 flags. The
  margin is still 0.2 (PASS).

## For review

**Constants changed from the PRD starting constants: none.** Every constant in
the PRD table was kept after looking at the region's spread (each Facet section
gives the reason). Where the PRD left a value open, this is how it was set:

| Constant | Value | Reason |
| --- | --- | --- |
| Weather: solar radiation D | 5,000 kJ m⁻² day⁻¹ | The PRD says "from the region's spread". This is the 5th–95th percentile spread of the June–August regime (17,779–22,671). The same D is kept for single months, so winter solar scores sit higher (January's spread is 2,011). |
| SolarAspect: sampled points | about 32 vertices per DayHike, counted from the nearer end of the walk | Cheap on the region, and the same set of points in both walking directions, so the reversed check is exact. |
| SolarAspect: east and north offsets | 2 FABDEM cells (2 arc-seconds, about 44 m east and 62 m north) | The PRD says "small offsets". A 30 m offset often reads the same 31 m cell; two cells always read a different cell at a known distance. |
| SolarAspect: slope floor | 5° | The PRD says "a few degrees". One or two metres of FABDEM noise over 44–62 m is 1–3°. |
| Corridor: densify step | radius / 10 (10 m for the 100 m Corridor) | Distance is measured to the line, and the error is under 0.2% of the radius. |
| Profile: DTW band | round(0.1 × 64) = 6 points | The PRD's 10% band, rounded to whole points. |

**Profile method kept: lock-step.** The rule was to keep DTW only if it
separates the verification pairs better, and it does not. The margin (lowest
similar minus highest dissimilar) is 0.173 for lock-step against 0.161 for DTW,
and the mean gap is 0.307 against 0.255. DTW lifts the dissimilar pairs more
than the similar ones, and it moves the random-pair median from 0.77 to 0.86.

**Rules chosen in this phase**

- Elevation gain is direction-free: max(climb, descent) along the walk,
  instead of the Catalog's `elevation_gain_m` in stored direction. OSM way order
  is arbitrary, so the stored figure fails the reversed check (fixture Ridge
  Walk: 450 m forward, 250 m reversed).
- The water count is capped at 3 in the Water measure. The fact is the
  true count.
- TarmacApproach on a loop is the shorter of the two ends. `_link` highways
  count as roads. An unlisted `surface` falls back to the highway kind.
- `presence` drops a flag that is missing on either side.
- RoundTrip is the main walk ending within `GAP_JOIN_M` (50 m) of its
  Trailhead. A lollipop is not one: the Hiker walks its stem back.
- Steepness reads elevation resampled every ~50 m, not raw vertices, so
  vertex spacing and DEM jitter between close vertices do not move it.
  ClimbGrade divides by the climbing length only: plain gain / length halves a
  walk that goes up and straight back down (in the region it is a median 0.66
  of ClimbGrade on RoundTrips against 0.89 on other DayHikes).

**Open design questions**

- **FABDEM vertex gain may be inflated by noise.** "Vršič - Razor" shows
  1,997 m of gain on a 902 m range. Smoothing the ElevationProfile before gain
  (and before ProfileShape) is not done yet. On a 50 m resample (as
  Steepness uses) the region's gain is a median 83% of the raw-vertex gain
  (`docs/future-work.md`, "Smoothed elevation gain").
- **Southness on mostly flat walks rests on a few sloped points** (embankments,
  road cuts). A minimum count of sloped points is an option if this turns out
  noisy in Ranking.
- **Only mapped water counts.** Springs, taps and water points are
  WaterSources since the re-ingest; streams, troughs and unmapped springs are
  not, so Water can still understate the water on backcountry hikes.
- **Which gain does the Catalog keep?** The direction-free gain (max of climb
  and descent) differs from the Catalog's stored-direction value on
  point-to-point hikes stored downhill. The promoted Catalog column has to pick
  one.
- **Lock-step over DTW.** Kept on the six verification pairs. Revisit if
  hikes whose top sits at a different point along the walk should match.
- **Presence is lenient** (both-absent agrees) in Landscape and Facilities.
  The ranking demo still separates (only 1.3% of candidates match the Seed's
  five flags), so this is revisited only if FacetWeights need sharper
  separation.
- **DryStretch is the whole length for the 58% of hikes with no WaterSource**, so
  for them Water partly repeats Effort's length.
- **Length is one way on a line.** RoundTrip says whether `length_m` is the
  whole walk back to the car; Effort still compares one-way lengths for the
  91.6% that are lines or lollipops (`docs/future-work.md`, "Walking back on
  a line").